In [1]:
import heapq
import random
from bisect import bisect_left, insort
from collections import Counter, defaultdict, deque
from math import gcd

# 1) Greedy

**In plain English.** A greedy algorithm makes the choice that looks best right now and
never takes it back. It is correct only when you can argue that some best solution makes
the same first choice.

> **Analogy.** Paying with the fewest banknotes: hand over the biggest note that fits,
> then repeat. It works for euros because of how the notes are sized — and it fails for
> strange coin systems, which is exactly why greedy needs a proof.

**The exchange argument.** Take any optimal solution that differs from yours at the first
choice; show you can swap in your choice without making it worse. If you can't find that
argument in a minute, the problem is probably DP.

**Recognition signals**

- "minimum number of jumps / boats / arrows / patches", "maximum number of …"
- a natural ordering: sort by deadline, end time, ratio, difference
- "can you reach / complete" with one running quantity (reach, tank, balance)
- two constraints from both sides (Candy) → two passes

**Common bugs**

- Using greedy where a counterexample exists (coin change with `[1, 3, 4]`).
- Sorting by the wrong key (by start instead of end, by value instead of difference).
- Forgetting the global feasibility check (Gas Station total, Hand of Straights length).

**Don't confuse with**

| Pattern | Difference |
|---|---|
| DP | tries all choices and keeps the best; greedy commits to one |
| Two pointers | often *implements* a greedy pairing (boats, cookies) |
| Heap-based greedy | when the best choice changes as items become available (IPO, scheduling) |

**Learning path.** Running reach (Jump Game) → BFS-like levels (Jump Game II) → restart
when the running total fails (Gas Station) → stretching a window to last occurrences
(Partition Labels) → a range of possible states (Valid Parenthesis String) → smallest
item forces a group (Hand of Straights) → filtering then covering (Triplets) → two passes
(Candy) → sort by difference (Two City) → matching sorted lists (Cookies) → local
placement (Flowers) → prefer the less flexible resource (Lemonade) → insertion order
(Queue) → coverage doubling (Patching Array).

## 1.1 Reachability and scans

Each question keeps one or two running quantities (reach, tank, balance, smallest missing
value) and updates them in a single pass; the work is in proving the greedy step is safe.

### 1.1.1 [55. Jump Game](https://leetcode.com/problems/jump-game/) — ★★★ · Medium

**Problem.** `nums[i]` is the max jump from `i`; can you reach the last index from index 0?  
**Signal.** Track the farthest reachable index.  
*New here: One running quantity (the farthest reach) replaces a search.*

**Fine print.** Ask before coding:
- Can values be 0? → Yes, and a 0 is how you get stuck.
- Is jumping fewer than `nums[i]` steps allowed? → Yes, `nums[i]` is the maximum.

> 💡 **Key insight.** Every index up to the farthest reach is reachable, so you only need to know whether the scan ever passes beyond it.

**Brute force →** DFS/BFS over jumps → O(n²). DP "reachable[i]" → O(n²). Running reach → O(n).

**Steps**
1. `reach = 0`.
2. For each `i`, if `i > reach`, return `False`.
3. Otherwise `reach = max(reach, i + nums[i])`.
4. Finishing the loop → `True`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [2]:
def can_jump(nums: list[int]) -> bool:
    """True if the last index is reachable."""
    # every index up to reach can be reached
    reach = 0
    for i, step in enumerate(nums):
        # a gap: index i can never be reached
        if i > reach:
            return False
        reach = max(reach, i + step)
    return True


assert can_jump([2, 3, 1, 1, 4])
assert not can_jump([3, 2, 1, 0, 4])
print(can_jump([2, 3, 1, 1, 4]))

True


### 1.1.2 [45. Jump Game II](https://leetcode.com/problems/jump-game-ii/) — ★★ · Medium

**Problem.** Minimum number of jumps to reach the last index (always reachable).  
**Signal.** Implicit BFS by levels: each jump extends the window of reachable indices.  
*New here: BFS levels without a queue: windows of indices.*

**Fine print.** Ask before coding:
- Is the end always reachable? → Yes, guaranteed.
- Single element? → 0 jumps.

> 💡 **Key insight.** All indices reachable with `j` jumps form a contiguous window; the next window ends at the farthest point reachable from inside it.

**Brute force →** DP `jumps[i] = 1 + min(jumps[j])` → O(n²). Window levels → O(n).

**Steps**
1. `end` = edge of the current jump's window; `far` = farthest reachable from it.
2. For each `i` (except the last), update `far`.
3. When `i == end`, take a jump: `jumps += 1`, `end = far`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [3]:
def jump(nums: list[int]) -> int:
    """Fewest jumps to reach the end."""
    jumps = end = far = 0
    # the last index never needs a jump from it
    for i in range(len(nums) - 1):
        far = max(far, i + nums[i])
        # leaving the current window costs one more jump
        if i == end:
            jumps += 1
            end = far
    return jumps


assert jump([2, 3, 1, 1, 4]) == 2
assert jump([2, 3, 0, 1, 4]) == 2
assert jump([0]) == 0
print(jump([2, 3, 1, 1, 4]))

2


### 1.1.3 [134. Gas Station](https://leetcode.com/problems/gas-station/) — ★★ · Medium

**Problem.** Circular route; `gas[i]` fuel at station `i`, `cost[i]` to drive to the next. Starting index for a full loop, or −1.  
**Signal.** If total gas ≥ total cost a start exists; a failed run rules out every start inside it.  
*New here: Restarting a scan and skipping a whole block of candidates at once.*

**Fine print.** Ask before coding:
- Is the answer unique if it exists? → Yes, guaranteed.
- Does the tank start empty and have no capacity limit? → Yes.

> 💡 **Key insight.** If you start at `s` and run dry before `i + 1`, then starting anywhere between `s` and `i` is no better (you'd arrive with even less fuel), so the next candidate is `i + 1`.

**Brute force →** Simulate from every station → O(n²). One pass with restarts → O(n).

**Steps**
1. If total gas < total cost, return −1.
2. Walk once with `tank += gas − cost`.
3. When `tank < 0`, set `start = i + 1` and reset the tank.
4. Return `start`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [4]:
def can_complete_circuit(gas: list[int], cost: list[int]) -> int:
    """Start index for a full loop, or -1."""
    # not enough fuel in total: no start can work
    if sum(gas) < sum(cost):
        return -1
    start = tank = 0
    for i, (g, c) in enumerate(zip(gas, cost)):
        tank += g - c
        # every start from `start` to i fails here: try i + 1
        if tank < 0:
            start, tank = i + 1, 0
    return start


assert can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3
assert can_complete_circuit([2, 3, 4], [3, 4, 3]) == -1
print(can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]))

3


### 1.1.4 [763. Partition Labels](https://leetcode.com/problems/partition-labels/) — ★★ · Medium

**Problem.** Split `s` into as many parts as possible so each letter appears in at most one part; return part sizes.  
**Signal.** A part must extend to the last occurrence of every letter inside it.  
*New here: Stretching the current window to the last occurrence of what it contains.*

**Fine print.** Ask before coding:
- Return part sizes, not the strings? → Sizes.
- Lowercase only? → Yes.

> 💡 **Key insight.** Once a letter is in the current part, the part must reach that letter's last occurrence; when the scan reaches the stretched end, the part can close.

**Brute force →** Try cut positions and check letters on both sides → O(n²). Last-index stretching → O(n).

**Steps**
1. Record each letter's last index.
2. Scan, setting `end = max(end, last[ch])`.
3. When `i == end`, record the part size and start a new part.

**Complexity**

- **Time:** O(n).
- **Space:** O(26).

In [5]:
def partition_labels(s: str) -> list[int]:
    """Sizes of maximal-count partitions."""
    last = {ch: i for i, ch in enumerate(s)}
    out, start, end = [], 0, 0
    for i, ch in enumerate(s):
        # the part must include ch's last occurrence
        end = max(end, last[ch])
        # every letter seen so far ends here: close the part
        if i == end:
            out.append(end - start + 1)
            start = i + 1
    return out


assert partition_labels("ababcbacadefegdehijhklij") == [9, 7, 8]
assert partition_labels("eccbbbbdec") == [10]
print(partition_labels("ababcbacadefegdehijhklij"))

[9, 7, 8]


### 1.1.5 [678. Valid Parenthesis String](https://leetcode.com/problems/valid-parenthesis-string/) — ★★ · Medium

**Problem.** `(`, `)` and `*` (which can be `(`, `)` or empty); is the string valid?  
**Signal.** Track the *range* of possible open counts.  
*New here: Tracking a *range* of possible states instead of one value.*

**Fine print.** Ask before coding:
- Can `*` be `(`, `)` or empty? → Yes, any of the three.
- Is an empty string valid? → Yes.

> 💡 **Key insight.** You don't need to decide what each `*` is; just keep the minimum and maximum possible number of open brackets.

**Brute force →** Try all 3^k assignments of stars → exponential. DP over (index, open count) → O(n²). Range → O(n).

**Steps**
1. `lo`, `hi` = min and max open count.
2. `(` → both +1; `)` → both −1; `*` → `lo − 1`, `hi + 1`.
3. `hi < 0` → invalid; clamp `lo` at 0.
4. Valid iff `lo == 0` at the end.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [6]:
def check_valid_string(s: str) -> bool:
    """True if '*' can be assigned to balance s."""
    # range of possible unmatched "(" counts
    lo = hi = 0
    for ch in s:
        lo += 1 if ch == "(" else -1
        hi += -1 if ch == ")" else 1
        # even treating every star as "(" is not enough
        if hi < 0:
            return False
        # open count can't go below zero
        lo = max(lo, 0)
    return lo == 0


assert check_valid_string("()")
assert check_valid_string("(*)")
assert check_valid_string("(*))")
assert not check_valid_string("(((*)")
print(check_valid_string("(*))"))

True


### 1.1.6 [846. Hand of Straights](https://leetcode.com/problems/hand-of-straights/) — ★★ · Medium

**Problem.** Can the cards be split into groups of `group_size` consecutive values?  
**Signal.** The smallest remaining card must start a group.  
*New here: A forced choice: the smallest item has no other option.*

**Fine print.** Ask before coding:
- Can `group_size` be 1? → Yes, then it is always `True`.
- Does `n % group_size != 0` mean impossible? → Yes, check it first.

> 💡 **Key insight.** The smallest remaining card cannot be in the middle of any group, so it must start one — no choice is involved.

**Brute force →** Backtracking over groupings → exponential. Forced groups from the smallest value → O(n log n).

**Steps**
1. Total must divide by the group size.
2. Count cards and visit values in sorted order.
3. If value `x` has `need` copies left, each must start a group: subtract `need` from `x, x+1, …`.
4. Any shortage → `False`.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(n).

In [7]:
def is_n_straight_hand(hand: list[int], group_size: int) -> bool:
    """True if hand splits into consecutive groups."""
    if len(hand) % group_size:
        return False
    count = Counter(hand)
    for x in sorted(count):
        # remaining copies of the smallest value must each start a group
        need = count[x]
        if need:
            for v in range(x, x + group_size):
                if count[v] < need:
                    return False
                count[v] -= need
    return True


assert is_n_straight_hand([1, 2, 3, 6, 2, 3, 4, 7, 8], 3)
assert not is_n_straight_hand([1, 2, 3, 4, 5], 4)
print(is_n_straight_hand([1, 2, 3, 6, 2, 3, 4, 7, 8], 3))

True


### 1.1.7 [1899. Merge Triplets to Form Target Triplet](https://leetcode.com/problems/merge-triplets-to-form-target-triplet/) — ★★ · Medium

**Problem.** Merging two triplets takes the element-wise max. Can you obtain `target`?  
**Signal.** Only triplets that never exceed the target in any coordinate are usable.  
*New here: Greedy filtering: discard anything that can only hurt.*

**Fine print.** Ask before coding:
- Is a merge an element-wise max? → Yes.
- Can any number of triplets be merged? → Yes.

> 💡 **Key insight.** Max never decreases, so a triplet exceeding the target anywhere is poison; merging *all* safe triplets is best, and it works iff each target coordinate is hit by some safe triplet.

**Brute force →** Try subsets of triplets → exponential. Filter + coverage → O(n).

**Steps**
1. Skip triplets with any component above the target.
2. For each safe triplet, note which coordinates equal the target.
3. Return whether all three coordinates were hit.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [8]:
def merge_triplets(triplets: list[list[int]], target: list[int]) -> bool:
    """True if target is reachable by max-merges."""
    # target coordinates matched by some safe triplet
    hit = set()
    for t in triplets:
        # a component above the target could never be undone
        if all(a <= b for a, b in zip(t, target)):
            hit.update(i for i in range(3) if t[i] == target[i])
    return len(hit) == 3


assert merge_triplets([[2, 5, 3], [1, 8, 4], [1, 7, 5]], [2, 7, 5])
assert not merge_triplets([[3, 4, 5], [4, 5, 6]], [3, 2, 5])
print(merge_triplets([[2, 5, 3], [1, 8, 4], [1, 7, 5]], [2, 7, 5]))

True


### 1.1.8 [135. Candy](https://leetcode.com/problems/candy/) — ★★ · Hard

**Problem.** Each child gets ≥ 1 candy; a child with a higher rating than a neighbour gets more than that neighbour. Minimum total?  
**Signal.** Two passes: satisfy left neighbours, then right neighbours.  
*New here: Two passes for constraints coming from both sides.*

**Fine print.** Ask before coding:
- Is it compared only with direct neighbours? → Yes.
- Can equal ratings get different amounts? → Yes, equal ratings have no constraint.

> 💡 **Key insight.** Each child must exceed a lower-rated left neighbour and a lower-rated right neighbour; satisfy each side in its own pass and take the max.

**Brute force →** Repeatedly fix violations until stable → O(n²). Two passes → O(n).

**Steps**
1. Everyone starts with 1.
2. Left to right: if rating rises, `candy[i] = candy[i − 1] + 1`.
3. Right to left: if rating rises leftward, `candy[i] = max(candy[i], candy[i + 1] + 1)`.
4. Sum.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [9]:
def candy(ratings: list[int]) -> int:
    """Minimum candies satisfying neighbour constraints."""
    n = len(ratings)
    out = [1] * n
    # satisfy every left neighbour
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            out[i] = out[i - 1] + 1
    # satisfy every right neighbour without breaking the left side
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            out[i] = max(out[i], out[i + 1] + 1)
    return sum(out)


assert candy([1, 0, 2]) == 5
assert candy([1, 2, 2]) == 4
print(candy([1, 0, 2]))

5


### 1.1.9 [1029. Two City Scheduling](https://leetcode.com/problems/two-city-scheduling/) — ★ · Medium

**Problem.** `2n` people, `costs[i] = [a, b]`; send exactly `n` to each city at minimum cost.  
**Signal.** Sort by the *saving* of choosing city A over B.  
*New here: Sorting by a *difference* rather than a raw value.*

**Fine print.** Ask before coding:
- Exactly `n` people per city? → Yes.
- Even `2n` total? → Yes.

> 💡 **Key insight.** Send everyone to B, then switching person `i` to A changes the cost by `a − b`; pick the n most negative changes.

**Brute force →** Try all C(2n, n) splits → exponential. DP → O(n²). Sort by difference → O(n log n).

**Steps**
1. Sort people by `a − b`.
2. The first n go to A, the rest to B.
3. Sum the chosen costs.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(n).

In [10]:
def two_city_sched_cost(costs: list[list[int]]) -> int:
    """Cheapest balanced assignment."""
    # most to gain from city A first
    costs = sorted(costs, key=lambda c: c[0] - c[1])
    n = len(costs) // 2
    return sum(a for a, _ in costs[:n]) + sum(b for _, b in costs[n:])


assert two_city_sched_cost([[10, 20], [30, 200], [400, 50], [30, 20]]) == 110
print(two_city_sched_cost([[10, 20], [30, 200], [400, 50], [30, 20]]))

110


### 1.1.10 [455. Assign Cookies](https://leetcode.com/problems/assign-cookies/) — ★ · Easy

**Problem.** Child `i` is content with a cookie of size ≥ `g[i]`. Maximise content children.  
**Signal.** Sort both; give each child the smallest cookie that satisfies them.  
*New here: Greedy matching of two sorted lists.*

**Fine print.** Ask before coding:
- At most one cookie per child? → Yes.
- Must a cookie be ≥ the greed factor? → Yes.

> 💡 **Key insight.** The least greedy child should get the smallest cookie that satisfies them; a cookie too small for them is too small for everyone.

**Brute force →** Try all assignments → exponential. Sort + two pointers → O(n log n).

**Steps**
1. Sort greed factors and cookie sizes.
2. For each cookie (small to large), if it satisfies the current child, move to the next child.
3. Return the number of satisfied children.

**Complexity**

- **Time:** O(n log n + m log m).
- **Space:** O(n + m) for the sorted copies.

In [11]:
def find_content_children(g: list[int], s: list[int]) -> int:
    """Number of satisfied children."""
    g, s = sorted(g), sorted(s)
    # index of the least greedy unsatisfied child
    child = 0
    for size in s:
        # smallest cookie that works goes to this child; smaller ones are useless
        if child < len(g) and size >= g[child]:
            child += 1
    return child


assert find_content_children([1, 2, 3], [1, 1]) == 1
assert find_content_children([1, 2], [1, 2, 3]) == 2
print(find_content_children([1, 2, 3], [1, 1]))

1


### 1.1.11 [605. Can Place Flowers](https://leetcode.com/problems/can-place-flowers/) — ★ · Easy

**Problem.** Flowerbed of 0/1, no adjacent flowers; can `n` new flowers be planted?  
**Signal.** Plant greedily at every legal empty spot from the left.  
*New here: Padding the array to remove edge cases.*

**Fine print.** Ask before coding:
- Is the existing bed valid (no adjacent flowers)? → Yes.
- Can `n` be 0? → Yes, then `True`.

> 💡 **Key insight.** Planting at the leftmost legal spot never blocks more future spots than skipping it would.

**Brute force →** Try all placements → exponential. Left-to-right greedy → O(n).

**Steps**
1. Pad the bed with a 0 on each side.
2. Plant wherever the cell and both neighbours are 0.
3. Count plantings; compare with `n`.

**Complexity**

- **Time:** O(n).
- **Space:** O(n) for the padded copy.

In [12]:
def can_place_flowers(flowerbed: list[int], n: int) -> bool:
    """True if n flowers fit without adjacency."""
    # zero padding makes the ends behave like the middle
    bed = [0] + flowerbed + [0]
    for i in range(1, len(bed) - 1):
        # plant as early as possible
        if bed[i - 1] == bed[i] == bed[i + 1] == 0:
            bed[i] = 1
            n -= 1
    return n <= 0


assert can_place_flowers([1, 0, 0, 0, 1], 1)
assert not can_place_flowers([1, 0, 0, 0, 1], 2)
print(can_place_flowers([0, 0, 1, 0, 0], 2))

True


### 1.1.12 [860. Lemonade Change](https://leetcode.com/problems/lemonade-change/) — ★ · Easy

**Problem.** Lemonade costs 5; customers pay 5, 10 or 20 in order. Can you always give change?  
**Signal.** Simulate, preferring to give a 10 as change for a 20 (keep 5s, which are more flexible).  
*New here: Greedy choice of which resource to spend.*

**Fine print.** Ask before coding:
- Do you start with no change? → Yes.
- Must customers be served in order? → Yes.

> 💡 **Key insight.** A 5 can make change for both 10s and 20s, a 10 only for 20s, so spend 10s first.

**Brute force →** Try both change options for every 20 → exponential. Preferring 10s → O(n).

**Steps**
1. Count 5s and 10s.
2. 5 → keep it; 10 → give back a 5.
3. 20 → give `10 + 5` if possible, else `5 + 5 + 5`.
4. Running out of 5s → `False`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [13]:
def lemonade_change(bills: list[int]) -> bool:
    """True if every customer gets correct change."""
    fives = tens = 0
    for b in bills:
        if b == 5:
            fives += 1
        elif b == 10:
            fives, tens = fives - 1, tens + 1
        # a 20: spend the less useful 10 first
        elif tens:
            fives, tens = fives - 1, tens - 1
        else:
            fives -= 3
        if fives < 0:
            return False
    return True


assert lemonade_change([5, 5, 5, 10, 20])
assert not lemonade_change([5, 5, 10, 10, 20])
print(lemonade_change([5, 5, 5, 10, 20]))

True


### 1.1.13 [406. Queue Reconstruction by Height](https://leetcode.com/problems/queue-reconstruction-by-height/) — ★ · Medium

**Problem.** People `[h, k]`: `k` people in front are at least as tall. Reconstruct the queue.  
**Signal.** Insert tallest first; shorter people do not affect taller people's `k`.  
*New here: Choosing an insertion order that makes later insertions harmless.*

**Fine print.** Ask before coding:
- Does `k` count people `>=` in height in front? → Yes.
- Is the input always valid? → Yes.

> 💡 **Key insight.** Shorter people are invisible to taller people's `k`, so place the tallest first; each newcomer is inserted at index `k` among people who are all at least as tall.

**Brute force →** Try permutations → factorial. Sorted insertion → O(n²).

**Steps**
1. Sort by height descending, then `k` ascending.
2. Insert each person at index `k` of the output.
3. Return the output.

**Complexity**

- **Time:** O(n²) — list insertions.
- **Space:** O(n).

In [14]:
def reconstruct_queue(people: list[list[int]]) -> list[list[int]]:
    """Queue consistent with every [h, k]."""
    out: list[list[int]] = []
    # tallest first; later (shorter) people never change earlier k values
    for p in sorted(people, key=lambda p: (-p[0], p[1])):
        out.insert(p[1], p)
    return out


people = [[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]]
assert reconstruct_queue(people) == [[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]
print(reconstruct_queue(people))

[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]


### 1.1.14 [330. Patching Array](https://leetcode.com/problems/patching-array/) — ★ · Hard

**Problem.** Sorted array; fewest numbers to add so every value in `1..n` is a sum of some subset.  
**Signal.** If all values in `[1, miss)` are coverable, adding `miss` extends coverage to `[1, 2·miss)`.  
*New here: Tracking coverage as an invariant `[1, miss)`.*

**Fine print.** Ask before coding:
- Is the array sorted? → Yes, ascending.
- Can `n` reach 2³¹ − 1? → Yes, so the covered reach can overflow 32 bits elsewhere.

> 💡 **Key insight.** If `[1, miss)` is covered, any number `x ≤ miss` extends coverage to `[1, miss + x)`; when the next number is too big, adding `miss` itself extends coverage the most.

**Brute force →** Track the set of all subset sums → exponential. Coverage invariant → O(m + log n).

**Steps**
1. `miss = 1` (nothing covered yet).
2. If the next array number ≤ `miss`, use it: `miss += num`.
3. Otherwise patch with `miss`: `miss *= 2`, count a patch.
4. Stop when `miss > n`.

**Complexity**

- **Time:** O(m + log n).
- **Space:** O(1).

In [15]:
def min_patches(nums: list[int], n: int) -> int:
    """Fewest additions covering 1..n."""
    # every value in [1, miss) is already a subset sum
    miss, i, patches = 1, 0, 0
    while miss <= n:
        # a small enough array number extends the coverage
        if i < len(nums) and nums[i] <= miss:
            miss += nums[i]
            i += 1
        # otherwise add miss itself, doubling the coverage
        else:
            miss += miss
            patches += 1
    return patches


assert min_patches([1, 3], 6) == 1
assert min_patches([1, 5, 10], 20) == 2
assert min_patches([1, 2, 2], 5) == 0
print(min_patches([1, 5, 10], 20))

2


---
# 2) Intervals

**In plain English.** Interval questions become easy once the intervals are sorted:
overlapping ones end up next to each other, and a single left-to-right pass decides
everything.

> **Analogy.** A day of meetings written on cards: lay the cards out by start time and
> overlapping meetings are side by side; to fit the *most* meetings, always pick the one
> that finishes earliest.

| Goal | Sort by | Why |
|---|---|---|
| merge / insert / union | start | overlapping intervals become adjacent |
| fewest removals, most non-overlapping, arrows | **end** | finishing earliest leaves the most room |
| max simultaneous overlap (rooms) | starts, with a min-heap of ends | count active intervals at each moment |

Two intervals `[a, b]` and `[c, d]` overlap iff `a <= d and c <= b`.

**Recognition signals**

- "merge", "insert", "overlap", "free time", "intersection"
- "minimum rooms / platforms / resources at once"
- "remove the fewest intervals", "fewest arrows / points that hit all"
- queries asking which interval contains a point → offline sort + heap

**Common bugs**

- Sorting by start when the problem needs the earliest end (scheduling).
- Touching endpoints: decide whether `[1, 2]` and `[2, 3]` overlap (problem-specific).
- Mutating the input's intervals while merging (copy if the caller needs them).
- Forgetting to extend with `max(end, …)` — a later interval can sit entirely inside.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Weighted interval scheduling | profits differ → DP + binary search, not greedy |
| Sweep line with events | counts overlaps by sorting +1/−1 events; a heap of ends does the same |
| Two pointers on sorted lists | used for intersecting two already sorted interval lists |

**Learning path.** Merge (sort by start) → insert (three phases) → fewest removals and
arrows (sort by end) → can one person attend (adjacent check) → rooms (heap of ends) →
offline queries (heap by size) → consecutive runs → two-list intersections → free time
(merge + gaps) → common slot → sorted calendar with binary search.

## 2.1 Merging and scheduling

First decide the sort key (start for merging, end for scheduling); then a single pass with
"the last kept interval" or a heap of active intervals answers the question.

### 2.1.1 [56. Merge Intervals](https://leetcode.com/problems/merge-intervals/) — ★★★ · Medium

**Problem.** Merge all overlapping intervals.  
**Signal.** Sort by start; overlapping intervals are then consecutive.  
*New here: Sorting by start so overlaps become adjacent.*

**Fine print.** Ask before coding:
- Is the input sorted? → No, sort by start.
- Do touching intervals (`[1,2]`, `[2,3]`) merge? → Yes.

> 💡 **Key insight.** After sorting by start, an interval overlaps the merged block iff it starts before that block ends.

**Brute force →** Compare every pair and merge until stable → O(n²) or worse. Sort + one pass → O(n log n).

**Steps**
1. Sort by start.
2. If the interval starts within the last merged block, extend its end with `max`.
3. Otherwise start a new block.

**Complexity**

- **Time:** O(n log n) — sorting.
- **Space:** O(n) for the output.

In [16]:
def merge_intervals(intervals: list[list[int]]) -> list[list[int]]:
    """Union of overlapping intervals."""
    out: list[list[int]] = []
    for start, end in sorted(intervals):
        # overlaps the last block: extend it (it may be fully inside)
        if out and start <= out[-1][1]:
            out[-1][1] = max(out[-1][1], end)
        else:
            out.append([start, end])
    return out


assert merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]) == [
    [1, 6],
    [8, 10],
    [15, 18],
]
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]]
print(merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]))

[[1, 6], [8, 10], [15, 18]]


### 2.1.2 [57. Insert Interval](https://leetcode.com/problems/insert-interval/) — ★★★ · Medium

**Problem.** Insert a new interval into sorted, non-overlapping intervals and merge.  
**Signal.** Three phases: before, overlapping, after.  
*New here: Splitting one pass into three phases.*

**Fine print.** Ask before coding:
- Sorted and non-overlapping input? → Yes.
- Return a new list? → Yes.

> 💡 **Key insight.** Intervals ending before the new one are untouched, intervals overlapping it fuse into one, and the rest are untouched — in that order.

**Brute force →** Append and run Merge Intervals → O(n log n). Three phases → O(n).

**Steps**
1. Copy intervals that end before the new one starts.
2. While intervals start before the new one ends, widen the new interval.
3. Append the widened interval, then the remaining intervals.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [17]:
def insert_interval(intervals: list[list[int]], new: list[int]) -> list[list[int]]:
    """Intervals with new inserted and merged."""
    out, i, n = [], 0, len(intervals)
    start, end = new
    # entirely before the new interval
    while i < n and intervals[i][1] < start:
        out.append(intervals[i])
        i += 1
    # overlapping: absorb into the new interval
    while i < n and intervals[i][0] <= end:
        start = min(start, intervals[i][0])
        end = max(end, intervals[i][1])
        i += 1
    out.append([start, end])
    # entirely after
    return out + intervals[i:]


assert insert_interval([[1, 3], [6, 9]], [2, 5]) == [[1, 5], [6, 9]]
assert insert_interval([[1, 2], [3, 5], [6, 7], [8, 10], [12, 16]], [4, 8]) == [
    [1, 2],
    [3, 10],
    [12, 16],
]
assert insert_interval([], [5, 7]) == [[5, 7]]
print(insert_interval([[1, 3], [6, 9]], [2, 5]))

[[1, 5], [6, 9]]


### 2.1.3 [435. Non-overlapping Intervals](https://leetcode.com/problems/non-overlapping-intervals/) — ★★★ · Medium

**Problem.** Fewest intervals to remove so the rest do not overlap.  
**Signal.** Keep the maximum number of non-overlapping intervals → sort by end (activity selection).  
*New here: Sorting by *end* and the exchange argument for scheduling.*

**Fine print.** Ask before coding:
- Do touching intervals overlap? → No, `[1,2]` and `[2,3]` are fine together.

> 💡 **Key insight.** Among intervals you could keep next, the one ending earliest leaves the most room; swapping any other choice for it never hurts.

**Brute force →** Try all subsets → O(2ⁿ). LIS-style DP → O(n²). Greedy by end → O(n log n).

**Steps**
1. Sort by end.
2. Keep an interval if it starts at or after the last kept end.
3. Removals = `n − kept`.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(1) besides sorting.

In [18]:
def erase_overlap_intervals(intervals: list[list[int]]) -> int:
    """Minimum removals to eliminate overlaps."""
    kept, last_end = 0, float("-inf")
    # earliest finishing first
    for start, end in sorted(intervals, key=lambda iv: iv[1]):
        # compatible with everything kept so far (touching is fine)
        if start >= last_end:
            kept += 1
            last_end = end
    return len(intervals) - kept


assert erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]) == 1
assert erase_overlap_intervals([[1, 2], [1, 2], [1, 2]]) == 2
assert erase_overlap_intervals([[1, 2], [2, 3]]) == 0
print(erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]))

1


### 2.1.4 [452. Minimum Number of Arrows to Burst Balloons](https://leetcode.com/problems/minimum-number-of-arrows-to-burst-balloons/) — ★★ · Medium

**Problem.** Balloons are intervals on the x-axis; an arrow at `x` bursts all containing `x`. Fewest arrows?  
**Signal.** Same as activity selection: shoot at the earliest end.  
*New here: The same greedy, counting groups instead of kept intervals.*

**Fine print.** Ask before coding:
- Does an arrow at an endpoint burst the balloon? → Yes, endpoints are inclusive.
- Can coordinates reach 32-bit limits? → Yes, so avoid computing differences that overflow.

> 💡 **Key insight.** The balloon that ends first must be hit by some arrow; shooting at its end bursts it and as many others as possible.

**Brute force →** Try arrow positions at every endpoint → O(n²). Greedy by end → O(n log n).

**Steps**
1. Sort by end.
2. If a balloon starts after the current arrow, shoot a new arrow at its end.
3. Count arrows.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(1) besides sorting.

In [19]:
def find_min_arrow_shots(points: list[list[int]]) -> int:
    """Fewest arrows to burst all balloons."""
    arrows, arrow_x = 0, float("-inf")
    for start, end in sorted(points, key=lambda p: p[1]):
        # missed by the last arrow: shoot at this balloon's end
        if start > arrow_x:
            arrows += 1
            arrow_x = end
    return arrows


assert find_min_arrow_shots([[10, 16], [2, 8], [1, 6], [7, 12]]) == 2
assert find_min_arrow_shots([[1, 2], [3, 4], [5, 6], [7, 8]]) == 4
assert find_min_arrow_shots([[1, 2], [2, 3], [3, 4], [4, 5]]) == 2
print(find_min_arrow_shots([[10, 16], [2, 8], [1, 6], [7, 12]]))

2


### 2.1.5 [252. Meeting Rooms](https://leetcode.com/problems/meeting-rooms/) — ★★★ · Easy · Premium

**Problem.** Can one person attend all meetings?  
**Signal.** Sort by start and check adjacent overlaps.  
*New here: Overlap detection needs only neighbours after sorting.*

**Fine print.** Ask before coding:
- Can one meeting end when the next starts? → Yes, that is fine.

> 💡 **Key insight.** In start order, if any two meetings overlap then some *adjacent* pair overlaps.

**Brute force →** Compare all pairs → O(n²). Sort + neighbours → O(n log n).

**Steps**
1. Sort by start.
2. Check every meeting ends by the time the next starts.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(n) for the sorted copy.

In [20]:
def can_attend_meetings(intervals: list[list[int]]) -> bool:
    """True if no meetings overlap."""
    intervals = sorted(intervals)
    # each meeting must end before the next one starts
    return all(a[1] <= b[0] for a, b in zip(intervals, intervals[1:]))


assert not can_attend_meetings([[0, 30], [5, 10], [15, 20]])
assert can_attend_meetings([[7, 10], [2, 4]])
print(can_attend_meetings([[7, 10], [2, 4]]))

True


### 2.1.6 [253. Meeting Rooms II](https://leetcode.com/problems/meeting-rooms-ii/) — ★★★ · Medium · Premium

**Problem.** Minimum number of meeting rooms needed.  
**Signal.** Max number of simultaneously active meetings → min-heap of end times.  
*New here: A heap of end times = the set of rooms in use.*

**Fine print.** Ask before coding:
- Does a meeting ending at `t` free a room for one starting at `t`? → Yes.
- Return only the count? → Yes.

> 💡 **Key insight.** When a meeting starts, the room that frees up first is the only one worth checking; reuse it if it is free, otherwise open a new room.

**Brute force →** Check overlaps at every time point → O(n²). Heap → O(n log n). (Two sorted lists of starts and ends also work.)

**Steps**
1. Sort by start.
2. If the earliest ending room is free by this start, reuse it (replace its end).
3. Otherwise push a new end (a new room).
4. The heap size is the number of rooms.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(n).

In [21]:
def min_meeting_rooms(intervals: list[list[int]]) -> int:
    """Rooms needed for all meetings."""
    # end times of rooms in use; earliest free room on top
    ends: list[int] = []
    for start, end in sorted(intervals):
        # earliest room is free: reuse it
        if ends and ends[0] <= start:
            heapq.heapreplace(ends, end)
        # every room is busy: open another
        else:
            heapq.heappush(ends, end)
    return len(ends)


assert min_meeting_rooms([[0, 30], [5, 10], [15, 20]]) == 2
assert min_meeting_rooms([[7, 10], [2, 4]]) == 1
print(min_meeting_rooms([[0, 30], [5, 10], [15, 20]]))

2


### 2.1.7 [1851. Minimum Interval to Include Each Query](https://leetcode.com/problems/minimum-interval-to-include-each-query/) — ★★ · Hard

**Problem.** For each query point, the size of the smallest interval containing it (−1 if none).  
**Signal.** Offline: sort queries and intervals, sweep with a min-heap keyed by size.  
*New here: Answering queries *offline* in sorted order.*

**Fine print.** Ask before coding:
- Are intervals inclusive? → Yes.
- Return answers in the original query order? → Yes, so sort the queries with their indices.

> 💡 **Key insight.** Sweeping queries left to right, intervals become available when their start passes and expire when their end passes, so a heap by size (with lazy removal of expired ones) gives each answer.

**Brute force →** Scan all intervals per query → O(n · q). Offline sweep → O((n + q) log n).

**Steps**
1. Sort intervals by start; process distinct queries in increasing order.
2. Push every interval with start ≤ query as `(size, end)`.
3. Pop tops that end before the query (they will never be useful again).
4. The top's size is the answer (or −1); map answers back to the original order.

**Complexity**

- **Time:** O((n + q) log n).
- **Space:** O(n + q).

In [22]:
def min_interval(intervals: list[list[int]], queries: list[int]) -> list[int]:
    """Smallest containing interval size per query."""
    intervals = sorted(intervals)
    # (size, end) of intervals that have started
    heap: list[tuple[int, int]] = []
    answers: dict[int, int] = {}
    i = 0
    for q in sorted(set(queries)):
        # intervals starting by q become candidates
        while i < len(intervals) and intervals[i][0] <= q:
            lo, hi = intervals[i]
            heapq.heappush(heap, (hi - lo + 1, hi))
            i += 1
        # drop intervals that ended before q (later queries are larger)
        while heap and heap[0][1] < q:
            heapq.heappop(heap)
        answers[q] = heap[0][0] if heap else -1
    return [answers[q] for q in queries]


assert min_interval([[1, 4], [2, 4], [3, 6], [4, 4]], [2, 3, 4, 5]) == [3, 3, 1, 4]
assert min_interval([[2, 3], [2, 5], [1, 8], [20, 25]], [2, 19, 5, 22]) == [2, -1, 4, 6]
print(min_interval([[1, 4], [2, 4], [3, 6], [4, 4]], [2, 3, 4, 5]))

[3, 3, 1, 4]


### 2.1.8 [228. Summary Ranges](https://leetcode.com/problems/summary-ranges/) — ★★ · Easy

**Problem.** Sorted unique integers → list of ranges like `"0->2"`.  
**Signal.** Group consecutive runs.  
*New here: Building intervals from points.*

**Fine print.** Ask before coding:
- Sorted and unique? → Yes.
- A single number? → Write it as `"5"`, not `"5->5"`.

> 💡 **Key insight.** A range continues exactly while the next number is one more than the current.

**Brute force →** Already linear; the care is in formatting single-number ranges.

**Steps**
1. Start a run at `i`.
2. Extend `j` while `nums[j + 1] == nums[j] + 1`.
3. Emit `"a"` or `"a->b"`; continue at `j + 1`.

**Complexity**

- **Time:** O(n).
- **Space:** O(n) for the output.

In [23]:
def summary_ranges(nums: list[int]) -> list[str]:
    """Compact range strings."""
    out, i = [], 0
    while i < len(nums):
        # extend the run of consecutive numbers
        j = i
        while j + 1 < len(nums) and nums[j + 1] == nums[j] + 1:
            j += 1
        out.append(str(nums[i]) if i == j else f"{nums[i]}->{nums[j]}")
        i = j + 1
    return out


assert summary_ranges([0, 1, 2, 4, 5, 7]) == ["0->2", "4->5", "7"]
assert summary_ranges([0, 2, 3, 4, 6, 8, 9]) == ["0", "2->4", "6", "8->9"]
print(summary_ranges([0, 1, 2, 4, 5, 7]))

['0->2', '4->5', '7']


### 2.1.9 [986. Interval List Intersections](https://leetcode.com/problems/interval-list-intersections/) — ★ · Medium

**Problem.** Intersections of two sorted, disjoint interval lists.  
**Signal.** Two pointers; advance the interval that ends first.  
*New here: Two pointers over two interval lists.*

**Fine print.** Ask before coding:
- Closed intervals? → Yes, so a single shared point counts.
- Is each list sorted and disjoint? → Yes.

> 💡 **Key insight.** The interval that ends first cannot intersect anything later in the other list, so after recording the overlap, advance it.

**Brute force →** Compare every pair → O(m · n). Two pointers → O(m + n).

**Steps**
1. Overlap of the current pair = `[max starts, min ends]` if non-empty.
2. Record it.
3. Advance the pointer whose interval ends first.

**Complexity**

- **Time:** O(m + n).
- **Space:** O(m + n) for the output.

In [24]:
def interval_intersection(a: list[list[int]], b: list[list[int]]) -> list[list[int]]:
    """Pairwise intersections of two interval lists."""
    out, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        # overlap of the current pair (if any)
        lo = max(a[i][0], b[j][0])
        hi = min(a[i][1], b[j][1])
        if lo <= hi:
            out.append([lo, hi])
        # the one ending first is finished
        if a[i][1] < b[j][1]:
            i += 1
        else:
            j += 1
    return out


a = [[0, 2], [5, 10], [13, 23], [24, 25]]
b = [[1, 5], [8, 12], [15, 24], [25, 26]]
assert interval_intersection(a, b) == [
    [1, 2],
    [5, 5],
    [8, 10],
    [15, 23],
    [24, 24],
    [25, 25],
]
print(interval_intersection(a, b))

[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


### 2.1.10 [759. Employee Free Time](https://leetcode.com/problems/employee-free-time/) — ★ · Hard · Premium

**Problem.** Each employee has sorted, disjoint working intervals; return the common free time.  
**Signal.** Merge all intervals, then report the gaps.  
*New here: Answering with the *gaps* of a merge.*

**Fine print.** Ask before coding:
- Positive-length gaps only? → Yes, skip zero-length gaps.
- Are free times bounded by the overall schedule, not infinite? → Yes.

> 💡 **Key insight.** Everyone is free exactly where nobody works, i.e. in the gaps of the union of all working intervals.

**Brute force →** Mark every time unit → depends on the time range. Sort + merge → O(N log N). (A k-way heap merge gives O(N log k).)

**Steps**
1. Flatten and sort all intervals by start.
2. Track the end of the merged block.
3. A start beyond that end creates a gap `[end, start]`.
4. Extend the end with `max`.

**Complexity**

- **Time:** O(N log N).
- **Space:** O(N).

In [25]:
def employee_free_time(schedule: list[list[list[int]]]) -> list[list[int]]:
    """Gaps where nobody works."""
    intervals = sorted(iv for emp in schedule for iv in emp)
    out, end = [], intervals[0][1]
    for s, e in intervals[1:]:
        # nobody works between the merged end and this start
        if s > end:
            out.append([end, s])
        end = max(end, e)
    return out


assert employee_free_time([[[1, 2], [5, 6]], [[1, 3]], [[4, 10]]]) == [[3, 4]]
assert employee_free_time([[[1, 3], [6, 7]], [[2, 4]], [[2, 5], [9, 12]]]) == [
    [5, 6],
    [7, 9],
]
print(employee_free_time([[[1, 2], [5, 6]], [[1, 3]], [[4, 10]]]))

[[3, 4]]


### 2.1.11 [1229. Meeting Scheduler](https://leetcode.com/problems/meeting-scheduler/) — ★ · Medium · Premium

**Problem.** Two people's free slots; the earliest common slot of length `duration`, or `[]`.  
**Signal.** Interval List Intersections, stopping at the first overlap long enough.  
*New here: Early exit on the first qualifying overlap.*

**Fine print.** Ask before coding:
- Do a person's own slots overlap? → No.
- Earliest by start time? → Yes.

> 💡 **Key insight.** Scanning both sorted lists with two pointers visits overlaps in time order, so the first long-enough one is the earliest.

**Brute force →** Compare all pairs → O(m · n). Sort + two pointers → O(m log m + n log n).

**Steps**
1. Sort both lists.
2. Overlap = `[max starts, min ends]`; if its length ≥ duration, return `[lo, lo + duration]`.
3. Advance the slot ending first.
4. No match → `[]`.

**Complexity**

- **Time:** O(m log m + n log n).
- **Space:** O(m + n) for the sorted copies.

In [26]:
def min_available_duration(
    slots1: list[list[int]], slots2: list[list[int]], duration: int
) -> list[int]:
    """Earliest common slot of the required length."""
    slots1, slots2 = sorted(slots1), sorted(slots2)
    i = j = 0
    while i < len(slots1) and j < len(slots2):
        lo = max(slots1[i][0], slots2[j][0])
        hi = min(slots1[i][1], slots2[j][1])
        # overlaps come in time order: the first fit is the earliest
        if hi - lo >= duration:
            return [lo, lo + duration]
        if slots1[i][1] < slots2[j][1]:
            i += 1
        else:
            j += 1
    return []


s1 = [[10, 50], [60, 120], [140, 210]]
s2 = [[0, 15], [60, 70]]
assert min_available_duration(s1, s2, 8) == [60, 68]
assert min_available_duration(s1, s2, 12) == []
print(min_available_duration(s1, s2, 8))

[60, 68]


### 2.1.12 [729. My Calendar I](https://leetcode.com/problems/my-calendar-i/) — ★ · Medium

**Problem.** `book(start, end)` succeeds if it does not overlap any existing booking.  
**Signal.** Keep bookings sorted; only the neighbours of the insertion point can overlap.  
*New here: Maintaining sorted intervals online with binary search.*

**Fine print.** Ask before coding:
- Half-open `[start, end)`? → Yes, so touching bookings are fine.
- Many calls? → Up to 1000, so a linear scan per call is fine.

> 💡 **Key insight.** Existing bookings don't overlap each other, so a new booking can only clash with the booking just before or just after its insertion point.

**Brute force →** Check every booking → O(n) per call. Binary search for the neighbours → O(log n) search (+ O(n) list insert).

**Steps**
1. Find the insertion index with `bisect_left`.
2. The previous booking must end by `start`.
3. The next booking must start at or after `end`.
4. If both hold, insert and return `True`.

**Complexity**

- **Time:** O(log n) search, O(n) insert (O(log n) with a balanced tree).
- **Space:** O(n).

In [27]:
class MyCalendar:
    """Non-overlapping booking calendar."""

    def __init__(self) -> None:
        # sorted, non-overlapping (start, end) pairs
        self.bookings: list[tuple[int, int]] = []

    def book(self, start: int, end: int) -> bool:
        i = bisect_left(self.bookings, (start, end))
        # clash with the booking before
        if i > 0 and self.bookings[i - 1][1] > start:
            return False
        # clash with the booking after
        if i < len(self.bookings) and self.bookings[i][0] < end:
            return False
        insort(self.bookings, (start, end))
        return True


cal = MyCalendar()
assert [cal.book(10, 20), cal.book(15, 25), cal.book(20, 30)] == [True, False, True]
print(cal.bookings)

[(10, 20), (20, 30)]


---
# 3) Math and Geometry

**In plain English.** Math questions hide a formula or an invariant. Simulating is usually
too slow or overflows; the trick is to see the structure (digits, factors, powers of two,
slopes) and compute it directly.

> **Analogy.** Counting the zeros at the end of 100! by multiplying it out is hopeless;
> noticing that each zero needs a factor 5 turns it into three divisions.

**Recognition signals**

- digits of a number: reverse, palindrome, plus one, digit sums
- huge `n` (up to 2³¹) → O(log n): fast power, repeated halving, counting factors
- "without using `*`, `/`, `%`" → bit shifts / repeated doubling
- points and lines → exact integer slopes with `gcd`
- "uniformly random" → Fisher–Yates

**Common bugs**

- 32-bit overflow rules (Python never overflows; clamp explicitly).
- Negative numbers: `-7 // 2 == -4` and `-7 % 10 == 3` in Python.
- Floating-point slopes: use reduced integer `(dy, dx)` pairs.
- Biased shuffle: picking `j` from the whole array instead of `0..i`.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Bit manipulation | operates on binary representation; math works on decimal digits and factors |
| DP | builds answers from subproblems; math finds a closed form |
| Simulation | follows the process step by step — often what the math replaces |

**Learning path.** Digit extraction (reverse, palindrome) → carry (plus one) → cycle
detection on a number sequence (happy) → fast power → grade-school multiplication →
counting factors (trailing zeros) → sieve → gcd on strings → binary long division. Then
geometry (slopes, squares) and randomness (Fisher–Yates).

## 3.1 Number manipulation

Core tools: `divmod(x, 10)` to peel digits, carries from the right, `gcd`, repeated
squaring / halving for O(log n), and counting prime factors instead of multiplying.

### 3.1.1 [7. Reverse Integer](https://leetcode.com/problems/reverse-integer/) — ★★ · Medium

**Problem.** Reverse the digits of a 32-bit signed integer; return 0 on overflow.  
**Signal.** Pop digits with `% 10` and `// 10`, check the 32-bit range.  
*New here: Digit peeling with `divmod`, and sign handling.*

**Fine print.** Ask before coding:
- Is the environment limited to 32-bit integers? → Yes, so check overflow before it happens.
- Negatives? → Keep the sign.

> 💡 **Key insight.** `divmod(x, 10)` removes the last digit and returns it; pushing it onto `rev` with `rev * 10 + digit` reverses the order.

**Brute force →** `int(str(abs(x))[::-1])` → works in Python; the digit loop is what fixed-width languages need.

**Steps**
1. Remember the sign and work with `abs(x)`.
2. Peel the last digit and append it to `rev`.
3. Restore the sign.
4. Out of 32-bit range → 0.

**Complexity**

- **Time:** O(digits).
- **Space:** O(1).

In [28]:
def reverse_integer(x: int) -> int:
    """Digit-reversed x, or 0 on 32-bit overflow."""
    sign = -1 if x < 0 else 1
    x, rev = abs(x), 0
    while x:
        # take the last digit off x and append it to rev
        x, digit = divmod(x, 10)
        rev = rev * 10 + digit
    rev *= sign
    return rev if -(2**31) <= rev <= 2**31 - 1 else 0


assert reverse_integer(123) == 321
assert reverse_integer(-123) == -321
assert reverse_integer(120) == 21
assert reverse_integer(1534236469) == 0
print(reverse_integer(-123))

-321


### 3.1.2 [9. Palindrome Number](https://leetcode.com/problems/palindrome-number/) — ★★ · Easy

**Problem.** Is an integer a palindrome, without converting to a string?  
**Signal.** Reverse only half the digits.  
*New here: Stopping halfway, when the reversed half catches up.*

**Fine print.** Ask before coding:
- Are negatives palindromes? → No.
- Without converting to a string? → That is the follow-up.

> 💡 **Key insight.** Reversing the second half and comparing it with the first half is enough, and it avoids overflow from reversing the whole number.

**Brute force →** `str(x) == str(x)[::-1]` → O(digits) with extra space. Half reversal → O(digits), O(1).

**Steps**
1. Negatives and numbers ending in 0 (except 0) → `False`.
2. Move digits from `x` to `rev` while `x > rev`.
3. Even length: `x == rev`; odd length: `x == rev // 10`.

**Complexity**

- **Time:** O(log x).
- **Space:** O(1).

In [29]:
def is_palindrome_number(x: int) -> bool:
    """True if x reads the same reversed."""
    # a trailing 0 would need a leading 0
    if x < 0 or (x % 10 == 0 and x != 0):
        return False
    rev = 0
    # stop when the reversed half reaches the remaining half
    while x > rev:
        x, digit = divmod(x, 10)
        rev = rev * 10 + digit
    # odd length: the middle digit sits at the end of rev
    return x == rev or x == rev // 10


assert is_palindrome_number(121)
assert not is_palindrome_number(-121)
assert not is_palindrome_number(10)
print(is_palindrome_number(12321))

True


### 3.1.3 [66. Plus One](https://leetcode.com/problems/plus-one/) — ★★ · Easy

**Problem.** Digits array of a number; add one.  
**Signal.** Carry from the end.  
*New here: Carry propagation with early exit.*

**Fine print.** Ask before coding:
- Most significant digit first? → Yes.
- No leading zeros? → Yes, except `[0]`.

> 💡 **Key insight.** Adding one only changes the trailing 9s (to 0) and the first non-9 digit (plus 1); if all digits are 9, the number grows by one digit.

**Brute force →** Convert to int, add, convert back → fine in Python, not in fixed width. Carry loop → O(n).

**Steps**
1. Walk from the last digit.
2. A digit < 9: increment and return.
3. A 9: set it to 0 and keep carrying.
4. All 9s: prepend 1.

**Complexity**

- **Time:** O(n).
- **Space:** O(n) for the copy.

In [30]:
def plus_one(digits: list[int]) -> list[int]:
    """Digits of the number plus one."""
    digits = digits.copy()
    for i in range(len(digits) - 1, -1, -1):
        # no carry needed beyond this digit
        if digits[i] < 9:
            digits[i] += 1
            return digits
        # 9 + 1 = 10: write 0, carry on
        digits[i] = 0
    # every digit was 9
    return [1] + digits


assert plus_one([1, 2, 3]) == [1, 2, 4]
assert plus_one([9, 9]) == [1, 0, 0]
print(plus_one([4, 3, 2, 1]))

[4, 3, 2, 2]


### 3.1.4 [202. Happy Number](https://leetcode.com/problems/happy-number/) — ★★ · Easy

**Problem.** Repeatedly replace `n` by the sum of squares of its digits; does it reach 1?  
**Signal.** The sequence either hits 1 or cycles → cycle detection.  
*New here: Floyd's cycle detection on a function instead of a linked list.*

**Fine print.** Ask before coding:
- Does a loop that never reaches 1 mean `False`? → Yes, detect it with a set or Floyd.

> 💡 **Key insight.** The sequence is a walk on numbers where each number has one successor, so it either reaches 1 or loops — exactly a linked list with or without a cycle.

**Brute force →** A set of seen numbers → O(steps) space. Floyd → O(1) space.

**Steps**
1. `nxt(x)` = sum of squared digits.
2. Slow moves one step, fast two steps.
3. Stop when fast is 1 (happy) or slow meets fast (cycle).

**Complexity**

- **Time:** O(log n) per step, few steps.
- **Space:** O(1).

In [31]:
def is_happy(n: int) -> bool:
    """True if n is a happy number."""

    def nxt(x: int) -> int:
        """Sum of squared digits."""
        return sum(int(d) ** 2 for d in str(x))

    # tortoise and hare on the sequence n, nxt(n), ...
    slow, fast = n, nxt(n)
    while fast != 1 and slow != fast:
        slow, fast = nxt(slow), nxt(nxt(fast))
    return fast == 1


assert is_happy(19)
assert not is_happy(2)
print(is_happy(19))

True


### 3.1.5 [50. Pow(x, n)](https://leetcode.com/problems/powx-n/) — ★★ · Medium

**Problem.** Compute `xⁿ` for integer `n` (possibly negative).  
**Signal.** Fast exponentiation by squaring.  
*New here: Exponentiation by squaring (reading `n` in binary).*

**Fine print.** Ask before coding:
- Negative `n`? → Return `1 / xⁿ`.
- Can `n` be −2³¹? → Yes, and negating it overflows 32 bits in other languages.

> 💡 **Key insight.** `xⁿ` is the product of `x^(2^k)` over the set bits `k` of `n`, and those powers come from squaring repeatedly.

**Brute force →** Multiply n times → O(n), too slow for n ≈ 2³¹. Squaring → O(log n).

**Steps**
1. Negative `n` → use `1/x` and `−n`.
2. While `n > 0`: if the lowest bit is set, multiply the result by `x`.
3. Square `x`, shift `n` right.

**Complexity**

- **Time:** O(log n).
- **Space:** O(1).

In [32]:
def my_pow(x: float, n: int) -> float:
    """x raised to n."""
    if n < 0:
        x, n = 1 / x, -n
    result = 1.0
    while n:
        # this bit of n contributes the current power of x
        if n & 1:
            result *= x
        # x, x^2, x^4, x^8, ...
        x *= x
        n >>= 1
    return result


assert abs(my_pow(2.0, 10) - 1024.0) < 1e-9
assert abs(my_pow(2.1, 3) - 9.261) < 1e-9
assert abs(my_pow(2.0, -2) - 0.25) < 1e-9
print(my_pow(2.0, 10))

1024.0


### 3.1.6 [43. Multiply Strings](https://leetcode.com/problems/multiply-strings/) — ★★ · Medium

**Problem.** Multiply two non-negative numbers given as strings, without big-int conversion.  
**Signal.** Grade-school multiplication: digit `i` × digit `j` lands in positions `i + j` and `i + j + 1`.  
*New here: Position arithmetic for digit products.*

**Fine print.** Ask before coding:
- No leading zeros except `"0"`? → Yes.
- Can I convert to int? → No.

> 💡 **Key insight.** The product of digit `i` and digit `j` contributes to result position `i + j + 1` with its carry going to `i + j`.

**Brute force →** Repeated addition → O(value). Grade-school → O(mn).

**Steps**
1. Result array of `m + n` zeros.
2. For each digit pair from the right, add the product to `res[i + j + 1]`.
3. Keep the ones digit there; add the carry to `res[i + j]`.
4. Join and strip leading zeros (empty → `"0"`).

**Complexity**

- **Time:** O(mn).
- **Space:** O(m + n).

In [33]:
def multiply(num1: str, num2: str) -> str:
    """Product of two decimal strings."""
    # an m-digit times an n-digit number has at most m + n digits
    res = [0] * (len(num1) + len(num2))
    for i in range(len(num1) - 1, -1, -1):
        for j in range(len(num2) - 1, -1, -1):
            # this digit product lands at i + j + 1, carrying into i + j
            total = int(num1[i]) * int(num2[j]) + res[i + j + 1]
            res[i + j + 1] = total % 10
            res[i + j] += total // 10
    out = "".join(map(str, res)).lstrip("0")
    return out or "0"


assert multiply("2", "3") == "6"
assert multiply("123", "456") == "56088"
assert multiply("0", "52") == "0"
print(multiply("123", "456"))

56088


### 3.1.7 [172. Factorial Trailing Zeroes](https://leetcode.com/problems/factorial-trailing-zeroes/) — ★★ · Medium

**Problem.** Number of trailing zeros in `n!`.  
**Signal.** Each trailing zero needs a factor 10 = 2 × 5, and 5s are rarer → count factors of 5.  
*New here: Counting prime factors instead of computing a product.*

**Fine print.** Ask before coding:
- O(log n)? → That is the follow-up.

> 💡 **Key insight.** Factors of 2 are plentiful, so the number of 10s equals the number of 5s: `n//5 + n//25 + n//125 + …`.

**Brute force →** Compute `n!` and count zeros → enormous numbers. Counting 5s → O(log n).

**Steps**
1. Repeatedly divide `n` by 5.
2. Add each quotient to the count.
3. Stop at 0.

**Complexity**

- **Time:** O(log n).
- **Space:** O(1).

In [34]:
def trailing_zeroes(n: int) -> int:
    """Trailing zeros of n factorial."""
    count = 0
    while n:
        # multiples of 5, then extra 5s from 25, 125, ...
        n //= 5
        count += n
    return count


assert trailing_zeroes(3) == 0
assert trailing_zeroes(5) == 1
assert trailing_zeroes(100) == 24
print(trailing_zeroes(100))

24


### 3.1.8 [204. Count Primes](https://leetcode.com/problems/count-primes/) — ★ · Medium

**Problem.** Number of primes strictly less than `n`.  
**Signal.** Sieve of Eratosthenes.  
*New here: The sieve.*

**Fine print.** Ask before coding:
- Strictly less than `n`? → Yes.
- `n ≤ 5·10⁶`? → Yes, so a sieve fits in memory.

> 💡 **Key insight.** Instead of testing each number, cross out the multiples of each prime; starting at `p²` is enough because smaller multiples were crossed out by smaller primes.

**Brute force →** Trial division for each number → O(n √n). Sieve → O(n log log n).

**Steps**
1. Mark everything prime except 0 and 1.
2. For each prime `p ≤ √n`, mark `p², p² + p, …` as composite.
3. Count the remaining primes.

**Complexity**

- **Time:** O(n log log n).
- **Space:** O(n).

In [35]:
def count_primes(n: int) -> int:
    """Primes below n."""
    if n < 3:
        return 0
    is_prime = [True] * n
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(n**0.5) + 1):
        if is_prime[p]:
            # smaller multiples were already crossed out by smaller primes
            is_prime[p * p :: p] = [False] * len(range(p * p, n, p))
    return sum(is_prime)


assert count_primes(10) == 4
assert count_primes(0) == 0
assert count_primes(1_000_000) == 78498
print(count_primes(10))

4


### 3.1.9 [1071. Greatest Common Divisor of Strings](https://leetcode.com/problems/greatest-common-divisor-of-strings/) — ★ · Easy

**Problem.** Largest string `x` that divides both `s` and `t` (each is `x` repeated).  
**Signal.** A common divisor exists iff `s + t == t + s`; its length is `gcd(len(s), len(t))`.  
*New here: Transferring gcd from numbers to strings.*

**Fine print.** Ask before coding:
- Must `x` divide both strings exactly? → Yes.
- None? → Return `""`.

> 💡 **Key insight.** If both strings are built from the same block, concatenating in either order gives the same string, and the largest block has length `gcd(|s|, |t|)`.

**Brute force →** Try every prefix length that divides both lengths → O((m + n) · divisors). The identity → O(m + n).

**Steps**
1. If `s + t != t + s`, return `""`.
2. Return the prefix of length `gcd(len(s), len(t))`.

**Complexity**

- **Time:** O(m + n).
- **Space:** O(m + n).

In [36]:
def gcd_of_strings(s: str, t: str) -> str:
    """Largest common divisor string."""
    # same block in both <=> concatenation order does not matter
    return s[: gcd(len(s), len(t))] if s + t == t + s else ""


assert gcd_of_strings("ABCABC", "ABC") == "ABC"
assert gcd_of_strings("ABABAB", "ABAB") == "AB"
assert gcd_of_strings("LEET", "CODE") == ""
print(gcd_of_strings("ABABAB", "ABAB"))

AB


### 3.1.10 [29. Divide Two Integers](https://leetcode.com/problems/divide-two-integers/) — ★ · Medium

**Problem.** Integer division truncating toward zero without `*`, `/` or `%`; clamp to 32-bit.  
**Signal.** Subtract shifted (doubled) divisors — long division in binary.  
*New here: Binary long division with shifts.*

**Fine print.** Ask before coding:
- Truncate toward zero? → Yes.
- `-2³¹ / -1`? → Clamp to 2³¹ − 1.
- Is the divisor non-zero? → Yes.

> 💡 **Key insight.** The quotient's bit `k` is 1 iff `divisor · 2^k` still fits in what is left of the dividend, checked from the highest bit down.

**Brute force →** Subtract the divisor repeatedly → O(quotient), up to 2³¹ steps. Shifted subtraction → O(32).

**Steps**
1. Record the sign; work with absolute values.
2. For `k` from 31 down to 0, if `divisor << k ≤ remaining`, subtract it and set bit `k`.
3. Apply the sign and clamp to the 32-bit range.

**Complexity**

- **Time:** O(32).
- **Space:** O(1).

In [37]:
def divide(dividend: int, divisor: int) -> int:
    """Truncated quotient within 32-bit range."""
    negative = (dividend < 0) != (divisor < 0)
    a, b = abs(dividend), abs(divisor)
    quotient = 0
    # largest multiples first, like long division
    for k in range(31, -1, -1):
        if (b << k) <= a:
            a -= b << k
            quotient |= 1 << k
    quotient = -quotient if negative else quotient
    # only -2^31 / -1 can overflow
    return max(-(2**31), min(2**31 - 1, quotient))


assert divide(10, 3) == 3
assert divide(7, -3) == -2
assert divide(-(2**31), -1) == 2**31 - 1
print(divide(10, 3))

3


## 3.2 Geometry and randomness

Geometry: represent directions exactly with reduced integer pairs, and fix one point (an
anchor or a corner) so the rest is a counting problem. Randomness: Fisher–Yates for a
uniform shuffle.

### 3.2.1 [149. Max Points on a Line](https://leetcode.com/problems/max-points-on-a-line/) — ★★ · Hard

**Problem.** Maximum number of points on one straight line.  
**Signal.** Fix an anchor point, group the others by slope.  
*New here: Exact slopes with gcd-reduced integer pairs.*

**Fine print.** Ask before coding:
- Unique points? → Yes on LeetCode; duplicates would need counting.
- Integer coordinates? → Yes, so use a reduced `(dy, dx)` key instead of a float slope.

> 💡 **Key insight.** Lines through a fixed anchor are identified by direction alone, so counting equal reduced directions gives the best line through that anchor.

**Brute force →** Every pair defines a line; check all points → O(n³). Anchor + slope counting → O(n²).

**Steps**
1. For each anchor, compute `(dx, dy)` to every later point.
2. Divide by `gcd` and fix the sign so equal directions get equal keys.
3. Count keys; best count + 1 (the anchor) is a candidate.
4. Return the best over all anchors.

**Complexity**

- **Time:** O(n² log C) — gcd per pair.
- **Space:** O(n).

In [38]:
def max_points(points: list[list[int]]) -> int:
    """Most points sharing a line."""
    if len(points) < 3:
        return len(points)
    best = 0
    for i, (x1, y1) in enumerate(points):
        slopes: Counter[tuple[int, int]] = Counter()
        for x2, y2 in points[i + 1 :]:
            dx, dy = x2 - x1, y2 - y1
            # reduce so parallel directions share one key
            g = gcd(dx, dy)
            dx, dy = dx // g, dy // g
            # canonical sign: (1, 2) and (-1, -2) are the same line
            if dx < 0 or (dx == 0 and dy < 0):
                dx, dy = -dx, -dy
            slopes[(dx, dy)] += 1
        # + 1 counts the anchor itself
        best = max(best, max(slopes.values(), default=0) + 1)
    return best


assert max_points([[1, 1], [2, 2], [3, 3]]) == 3
assert max_points([[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]) == 4
print(max_points([[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]))

4


### 3.2.2 [2013. Detect Squares](https://leetcode.com/problems/detect-squares/) — ★★ · Medium

**Problem.** `add(point)`; `count(point)` = number of axis-aligned squares with the query point as one corner and three added points as the others (duplicates count separately).  
**Signal.** Choose the diagonal corner; the other two corners are then determined.  
*New here: Fixing one free choice so everything else is determined.*

**Fine print.** Ask before coding:
- Do duplicate points each count? → Yes, multiply by their counts.
- Axis-aligned squares with positive area only? → Yes.

> 💡 **Key insight.** Once the corner diagonally opposite the query is chosen, the other two corners are forced, so multiply the counts of the three points.

**Brute force →** Try all triples of stored points → O(n³). Diagonal choice → O(distinct points).

**Steps**
1. Store point counts.
2. For each stored `(x, y)` with `|x − qx| == |y − qy| > 0` (a diagonal):
3. add `count(x, y) · count(qx, y) · count(x, qy)`.

**Complexity**

- **Time:** O(1) add, O(distinct points) count.
- **Space:** O(distinct points).

In [39]:
class DetectSquares:
    """Count axis-aligned squares through a query point."""

    def __init__(self) -> None:
        self.points: Counter[tuple[int, int]] = Counter()

    def add(self, point: list[int]) -> None:
        self.points[tuple(point)] += 1

    def count(self, point: list[int]) -> int:
        qx, qy = point
        total = 0
        for (x, y), n in self.points.items():
            # (x, y) is diagonal to the query: a non-degenerate square
            if abs(x - qx) == abs(y - qy) and x != qx:
                # the two remaining corners are forced
                total += n * self.points[(qx, y)] * self.points[(x, qy)]
        return total


ds = DetectSquares()
for p in [[3, 10], [11, 2], [3, 2]]:
    ds.add(p)
assert ds.count([11, 10]) == 1 and ds.count([14, 8]) == 0
ds.add([11, 2])
assert ds.count([11, 10]) == 2
print("ok")

ok


### 3.2.3 [384. Shuffle an Array](https://leetcode.com/problems/shuffle-an-array/) — ★ · Medium

**Problem.** `shuffle()` returns a uniformly random permutation; `reset()` restores the original.  
**Signal.** Fisher–Yates shuffle.  
*New here: Fisher–Yates and why its range matters.*

**Fine print.** Ask before coding:
- Must every permutation be equally likely? → Yes, so use Fisher–Yates.
- Does `reset` restore the original order? → Yes.

> 💡 **Key insight.** Filling positions from the back, each position picks uniformly from the elements not yet placed, so every permutation has probability `1/n!`.

**Brute force →** Sort by random keys → O(n log n). Swapping with any index `0..n−1` each time → fast but *biased*. Fisher–Yates → O(n), unbiased.

**Steps**
1. Copy the original.
2. For `i` from n − 1 down to 1, pick `j` uniformly in `0..i`.
3. Swap `a[i]` and `a[j]`.
4. `reset` returns a copy of the original.

**Complexity**

- **Time:** O(n) per shuffle.
- **Space:** O(n).

In [40]:
class Shuffler:
    """Uniform random shuffling with reset."""

    def __init__(self, nums: list[int]) -> None:
        self.original = nums.copy()

    def reset(self) -> list[int]:
        return self.original.copy()

    def shuffle(self) -> list[int]:
        a = self.original.copy()
        for i in range(len(a) - 1, 0, -1):
            # choose a[i] uniformly from the not-yet-placed prefix a[0..i]
            j = random.randint(0, i)
            a[i], a[j] = a[j], a[i]
        return a


random.seed(0)
sh = Shuffler([1, 2, 3])
counts = Counter(tuple(sh.shuffle()) for _ in range(6000))
assert len(counts) == 6 and min(counts.values()) > 850
assert sh.reset() == [1, 2, 3]
print(counts)

Counter({(3, 1, 2): 1027, (2, 3, 1): 1008, (2, 1, 3): 1003, (1, 3, 2): 993, (1, 2, 3): 989, (3, 2, 1): 980})


---
# 4) Bit Manipulation

**In plain English.** Bit manipulation treats a number as a row of on/off switches and uses
AND, OR, XOR and shifts to read or flip them directly — often replacing a hash map or a
loop with one operation.

> **Analogy.** A row of light switches: XOR is flipping a switch (flip it twice and it is
> back off), AND with a mask is looking at only some switches, shifting is sliding the
> whole row left or right.

| Trick | Meaning |
|---|---|
| `x & 1` | lowest bit (odd?) |
| `x & (x − 1)` | clear the lowest set bit (power of two ⇔ result is 0) |
| `x & −x` | isolate the lowest set bit |
| `a ^ a = 0`, `a ^ 0 = a` | XOR cancels pairs |
| `x >> k & 1` | k-th bit |

Python ints are unbounded, so emulate 32-bit behaviour with a mask `0xFFFFFFFF` where the
problem relies on overflow.

**Recognition signals**

- "every element appears twice / three times except one", "missing number"
- "O(1) extra space" on counting / pairing problems
- "number of 1 bits", "power of two", "reverse bits", "Hamming distance"
- "without using + / −" → XOR plus carry
- sets of ≤ 32 small items (letters) → bitmasks

**Common bugs**

- Negative numbers in Python: `>>` never reaches 0 for negatives and there is no 32-bit
  wrap — mask with `0xFFFFFFFF` and convert back from two's complement.
- Operator precedence: `a & b == 0` means `a & (b == 0)`; parenthesise.
- Forgetting `n > 0` in the power-of-two test.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Hash map counting | O(n) space; XOR tricks get O(1) for "appears twice" problems |
| Math (digits) | works in base 10; bit tricks work in base 2 |
| Bitmask DP / backtracking | uses bits as a *set*; here bits are the numbers themselves |

**Learning path.** XOR cancellation (Single Number) → per-bit counting mod 3 → splitting by
a differing bit (two singles) → XOR with indices (Missing Number) → clearing the lowest bit
(popcount) → DP over bits → shifting bits out and in (reverse) → addition with XOR and
carry → binary strings → common prefix (range AND) → power of two → XOR distance → bit-by-bit
decisions (min flips).

## 4.1 XOR and bit counting

Two families: XOR cancellation (pairs vanish, leaving what is unpaired) and per-bit
reasoning (look at one bit position at a time, across all numbers or across `a`, `b`, `c`).

### 4.1.1 [136. Single Number](https://leetcode.com/problems/single-number/) — ★★ · Easy

**Problem.** Every element appears twice except one; find it in O(1) space.  
**Signal.** XOR cancels pairs.  
*New here: XOR cancellation.*

**Fine print.** Ask before coding:
- Exactly one single element? → Yes.
- O(1) extra space? → Yes, so use XOR.

> 💡 **Key insight.** XOR is commutative and `a ^ a = 0`, so XOR-ing everything cancels every pair and leaves the single number.

**Brute force →** Hash-map counts → O(n) space. Sort and scan → O(n log n). XOR → O(n) time, O(1) space.

**Steps**
1. Start with 0.
2. XOR every number into it.
3. Return the result.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [41]:
def single_number(nums: list[int]) -> int:
    """The element without a pair."""
    out = 0
    for x in nums:
        # pairs cancel: a ^ a = 0
        out ^= x
    return out


assert single_number([2, 2, 1]) == 1
assert single_number([4, 1, 2, 1, 2]) == 4
print(single_number([4, 1, 2, 1, 2]))

4


### 4.1.2 [137. Single Number II](https://leetcode.com/problems/single-number-ii/) — ★★ · Medium

**Problem.** Every element appears three times except one; find it in O(1) space.  
**Signal.** Count each bit modulo 3.  
*New here: Per-bit counting with a modulus, and converting back from 32-bit two's complement.*

**Fine print.** Ask before coding:
- Negative values? → Yes; in Python, convert the 32-bit result back to a signed int.
- O(1) space? → Yes.

> 💡 **Key insight.** For each bit, numbers appearing three times contribute a multiple of 3 to the count, so `count % 3` is exactly the single number's bit.

**Brute force →** Hash-map counts → O(n) space. Per-bit counting → O(32 n) time, O(1) space.

**Steps**
1. For each of 32 bit positions, count how many numbers have that bit set.
2. If the count mod 3 is 1, set that bit in the result.
3. If bit 31 is set, convert to a negative number.

**Complexity**

- **Time:** O(32 n).
- **Space:** O(1).

In [42]:
def single_number_ii(nums: list[int]) -> int:
    """The element appearing once among triples."""
    out = 0
    for b in range(32):
        # triples contribute multiples of 3 to this bit's count
        if sum((x >> b) & 1 for x in nums) % 3:
            out |= 1 << b
    # bit 31 set means a negative 32-bit number
    return out - (1 << 32) if out >= 1 << 31 else out


assert single_number_ii([2, 2, 3, 2]) == 3
assert single_number_ii([0, 1, 0, 1, 0, 1, 99]) == 99
assert single_number_ii([-2, -2, 1, 1, 4, 1, 4, 4, -4, -2]) == -4
print(single_number_ii([0, 1, 0, 1, 0, 1, 99]))

99


### 4.1.3 [260. Single Number III](https://leetcode.com/problems/single-number-iii/) — ★ · Medium

**Problem.** Exactly two elements appear once, all others twice; find both.  
**Signal.** XOR all → `a ^ b`; split numbers by one bit where `a` and `b` differ.  
*New here: Splitting into two groups by a distinguishing bit.*

**Fine print.** Ask before coding:
- Any order? → Yes.
- O(1) space? → Yes.

> 💡 **Key insight.** `a ^ b` has a 1 wherever `a` and `b` differ; grouping all numbers by one such bit puts `a` and `b` in different groups, each with only pairs otherwise.

**Brute force →** Hash-map counts → O(n) space. XOR + split → O(n), O(1).

**Steps**
1. XOR everything → `diff = a ^ b`.
2. `low = diff & −diff` (lowest differing bit).
3. XOR the numbers with that bit set → `a`.
4. `b = diff ^ a`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [43]:
def single_number_iii(nums: list[int]) -> list[int]:
    """The two unpaired elements."""
    # pairs cancel, leaving a ^ b
    diff = 0
    for x in nums:
        diff ^= x
    # a bit where a and b differ
    low = diff & -diff
    # XOR of one group: pairs cancel, a remains
    a = 0
    for x in nums:
        if x & low:
            a ^= x
    return sorted([a, diff ^ a])


assert single_number_iii([1, 2, 1, 3, 2, 5]) == [3, 5]
assert single_number_iii([-1, 0]) == [-1, 0]
print(single_number_iii([1, 2, 1, 3, 2, 5]))

[3, 5]


### 4.1.4 [268. Missing Number](https://leetcode.com/problems/missing-number/) — ★★★ · Easy

**Problem.** `n` distinct numbers from `0..n`; find the missing one.  
**Signal.** XOR indices and values (or use the sum formula).  
*New here: Pairing values with indices via XOR.*

**Fine print.** Ask before coding:
- Values in `0..n`, distinct? → Yes.
- O(1) space? → Yes, by sum or XOR.

> 💡 **Key insight.** XOR-ing all of `0..n` with all the values cancels every number that is present, leaving the missing one.

**Brute force →** Set lookup for each candidate → O(n) space. Sort → O(n log n). Sum formula or XOR → O(n), O(1).

**Steps**
1. Start with `n` (the index that has no slot).
2. XOR in every index `i` and value `x`.
3. Return the result.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [44]:
def missing_number(nums: list[int]) -> int:
    """The value in 0..n not present."""
    # n has no index of its own, so seed with it
    out = len(nums)
    for i, x in enumerate(nums):
        # every present value cancels its matching index
        out ^= i ^ x
    return out


assert missing_number([3, 0, 1]) == 2
assert missing_number([0, 1]) == 2
assert missing_number([9, 6, 4, 2, 3, 5, 7, 0, 1]) == 8
print(missing_number([3, 0, 1]))

2


### 4.1.5 [191. Number of 1 Bits](https://leetcode.com/problems/number-of-1-bits/) — ★★★ · Easy

**Problem.** Count set bits (Hamming weight).  
**Signal.** `n & (n − 1)` drops the lowest set bit.  
*New here: The `n & (n − 1)` trick.*

**Fine print.** Ask before coding:
- Unsigned 32-bit input? → Yes.
- Follow-up: called many times? → Use a lookup table.

> 💡 **Key insight.** Subtracting 1 flips the lowest set bit and everything below it, so `n & (n − 1)` removes exactly one set bit.

**Brute force →** Check all 32 bits → O(32). Clearing set bits → O(number of set bits).

**Steps**
1. While `n` is non-zero, clear its lowest set bit.
2. Count the iterations.

**Complexity**

- **Time:** O(set bits).
- **Space:** O(1).

In [45]:
def hamming_weight(n: int) -> int:
    """Number of 1 bits."""
    count = 0
    while n:
        # removes exactly the lowest set bit
        n &= n - 1
        count += 1
    return count


assert hamming_weight(11) == 3
assert hamming_weight(128) == 1
assert hamming_weight(2147483645) == 30
print(hamming_weight(11))

3


### 4.1.6 [338. Counting Bits](https://leetcode.com/problems/counting-bits/) — ★★★ · Easy

**Problem.** For every `i` in `0..n`, the number of set bits.  
**Signal.** Reuse smaller answers: `bits[i] = bits[i >> 1] + (i & 1)`.  
*New here: DP over bit patterns.*

**Fine print.** Ask before coding:
- O(n) without counting each number separately? → Yes, reuse `bits[i >> 1]`.

> 💡 **Key insight.** `i >> 1` is `i` without its last bit, and it is smaller, so its count is already known; add back the last bit.

**Brute force →** Popcount every number → O(n log n). DP → O(n).

**Steps**
1. `bits[0] = 0`.
2. For each `i`, `bits[i] = bits[i >> 1] + (i & 1)`.

**Complexity**

- **Time:** O(n).
- **Space:** O(n) for the output.

In [46]:
def count_bits(n: int) -> list[int]:
    """Set-bit counts for 0..n."""
    bits = [0] * (n + 1)
    for i in range(1, n + 1):
        # i without its last bit, plus that last bit
        bits[i] = bits[i >> 1] + (i & 1)
    return bits


assert count_bits(2) == [0, 1, 1]
assert count_bits(5) == [0, 1, 1, 2, 1, 2]
print(count_bits(5))

[0, 1, 1, 2, 1, 2]


### 4.1.7 [190. Reverse Bits](https://leetcode.com/problems/reverse-bits/) — ★★★ · Easy

**Problem.** Reverse the bits of a 32-bit unsigned integer.  
**Signal.** Shift bits out of `n` and into the result, 32 times.  
*New here: Moving bits between two numbers with shifts.*

**Fine print.** Ask before coding:
- Exactly 32 bits, padded with leading zeros? → Yes.
- Follow-up: called many times? → Cache the reversed bytes.

> 💡 **Key insight.** Reading bits from the low end of `n` and pushing them onto the low end of `out` reverses their order — like reversing digits.

**Brute force →** Convert to a 32-character string and reverse → fine but slower. Shifts → O(32).

**Steps**
1. Repeat 32 times:
2. `out = (out << 1) | (n & 1)`.
3. `n >>= 1`.

**Complexity**

- **Time:** O(32).
- **Space:** O(1).

In [47]:
def reverse_bits(n: int) -> int:
    """32-bit reversal of n."""
    out = 0
    for _ in range(32):
        # take n's lowest bit, push it onto out
        out = (out << 1) | (n & 1)
        n >>= 1
    return out


assert reverse_bits(0b00000010100101000001111010011100) == 964176192
assert reverse_bits(0b11111111111111111111111111111101) == 3221225471
print(reverse_bits(43261596))

964176192


### 4.1.8 [371. Sum of Two Integers](https://leetcode.com/problems/sum-of-two-integers/) — ★★★ · Medium

**Problem.** Add two integers without `+` or `−`.  
**Signal.** XOR adds without carry; AND shifted left is the carry.  
*New here: Binary addition from XOR and AND, with 32-bit masking.*

**Fine print.** Ask before coding:
- Negative values? → Yes, so Python needs a 32-bit mask.

> 💡 **Key insight.** Adding bit by bit, `a ^ b` is the sum ignoring carries and `(a & b) << 1` is the carries; repeat until no carry remains.

**Brute force →** Increment one at a time → O(|b|). XOR/carry loop → O(32).

**Steps**
1. While there is a carry: `a, b = a ^ b, (a & b) << 1`, both masked to 32 bits.
2. Mask the result.
3. Convert from two's complement if bit 31 is set.

**Complexity**

- **Time:** O(32).
- **Space:** O(1).

⚠️ **Trap.** Without the mask, negative numbers loop forever in Python (the carry never runs off the end).

In [48]:
def get_sum(a: int, b: int) -> int:
    """a + b using only bit operations."""
    mask = 0xFFFFFFFF
    while b & mask:
        # sum without carries, and the carries shifted into place
        a, b = (a ^ b) & mask, ((a & b) << 1) & mask
    a &= mask
    # reinterpret as signed 32-bit
    return a if a < 1 << 31 else a - (1 << 32)


assert get_sum(1, 2) == 3
assert get_sum(2, 3) == 5
assert get_sum(-1, 1) == 0
assert get_sum(-12, -8) == -20
print(get_sum(-12, -8))

-20


### 4.1.9 [67. Add Binary](https://leetcode.com/problems/add-binary/) — ★★ · Easy

**Problem.** Sum of two binary strings as a binary string.  
**Signal.** Digit-by-digit addition with carry from the right.  
*New here: Grade-school addition in base 2.*

**Fine print.** Ask before coding:
- No leading zeros except `"0"`? → Yes.
- Can I use `int(a, 2)`? → Usually not.

> 💡 **Key insight.** Same as decimal addition with carry, but each column's digit is `total % 2` and the carry is `total // 2`.

**Brute force →** `bin(int(a, 2) + int(b, 2))` → fine in Python; the loop works for any length in any language.

**Steps**
1. Two pointers at the ends; carry 0.
2. Add available bits and the carry.
3. Emit `total % 2`; carry `total // 2`.
4. Reverse the output.

**Complexity**

- **Time:** O(max(m, n)).
- **Space:** O(max(m, n)).

In [49]:
def add_binary(a: str, b: str) -> str:
    """Binary sum of two binary strings."""
    i, j, carry = len(a) - 1, len(b) - 1, 0
    out = []
    # continue while digits or a carry remain
    while i >= 0 or j >= 0 or carry:
        total = carry
        if i >= 0:
            total += int(a[i])
            i -= 1
        if j >= 0:
            total += int(b[j])
            j -= 1
        out.append(str(total % 2))
        carry = total // 2
    return "".join(reversed(out))


assert add_binary("11", "1") == "100"
assert add_binary("1010", "1011") == "10101"
print(add_binary("1010", "1011"))

10101


### 4.1.10 [201. Bitwise AND of Numbers Range](https://leetcode.com/problems/bitwise-and-of-numbers-range/) — ★★ · Medium

**Problem.** Bitwise AND of all integers in `[left, right]`.  
**Signal.** The result is the common binary prefix of `left` and `right`.  
*New here: Reasoning about which bits must flip inside a range.*

**Fine print.** Ask before coding:
- `left ≤ right`? → Yes.
- `left = 0`? → The answer is 0.

> 💡 **Key insight.** Below the common prefix of `left` and `right`, some number in the range has a 0 in every position, so only the common prefix survives the AND.

**Brute force →** AND every number → O(right − left), up to 2³¹. Common prefix → O(32).

**Steps**
1. Shift both right until they are equal, counting shifts.
2. Shift the common prefix back left.

**Complexity**

- **Time:** O(32).
- **Space:** O(1).

In [50]:
def range_bitwise_and(left: int, right: int) -> int:
    """AND of all numbers in [left, right]."""
    shift = 0
    # drop differing low bits until only the common prefix is left
    while left != right:
        left >>= 1
        right >>= 1
        shift += 1
    return left << shift


assert range_bitwise_and(5, 7) == 4
assert range_bitwise_and(0, 0) == 0
assert range_bitwise_and(1, 2147483647) == 0
print(range_bitwise_and(5, 7))

4


### 4.1.11 [231. Power of Two](https://leetcode.com/problems/power-of-two/) — ★ · Easy

**Problem.** Is `n` a power of two?  
**Signal.** A power of two has exactly one set bit.  
*New here: A one-line structural test.*

**Fine print.** Ask before coding:
- Negative or 0? → `False`.

> 💡 **Key insight.** Powers of two have exactly one set bit, so clearing the lowest set bit leaves 0.

**Brute force →** Divide by 2 repeatedly → O(log n). Bit test → O(1).

**Steps**
1. `n` must be positive.
2. `n & (n − 1)` must be 0.

**Complexity**

- **Time:** O(1).
- **Space:** O(1).

In [51]:
def is_power_of_two(n: int) -> bool:
    """True if n is 2**k."""
    # exactly one set bit
    return n > 0 and n & (n - 1) == 0


assert is_power_of_two(1) and is_power_of_two(16)
assert not is_power_of_two(3) and not is_power_of_two(0)
print(is_power_of_two(1024))

True


### 4.1.12 [461. Hamming Distance](https://leetcode.com/problems/hamming-distance/) — ★ · Easy

**Problem.** Number of positions where the bits of `x` and `y` differ.  
**Signal.** XOR marks differing bits; count them.  
*New here: Combining XOR with popcount.*

**Fine print.** Ask before coding:
- Non-negative 32-bit integers? → Yes.

> 💡 **Key insight.** `x ^ y` has a 1 exactly where the bits differ.

**Brute force →** Compare 32 bit positions → O(32). XOR + popcount → O(set bits).

**Steps**
1. Compute `x ^ y`.
2. Count its set bits.

**Complexity**

- **Time:** O(32).
- **Space:** O(1).

In [52]:
def hamming_distance(x: int, y: int) -> int:
    """Differing bit count."""
    # 1 bits of x ^ y mark the differences
    return hamming_weight(x ^ y)


assert hamming_distance(1, 4) == 2
assert hamming_distance(3, 1) == 1
print(hamming_distance(1, 4))

2


### 4.1.13 [1318. Minimum Flips to Make a OR b Equal to c](https://leetcode.com/problems/minimum-flips-to-make-a-or-b-equal-to-c/) — ★ · Medium

**Problem.** Fewest bit flips in `a` and `b` so that `a | b == c`.  
**Signal.** Decide bit by bit.  
*New here: Independent per-bit decisions.*

**Fine print.** Ask before coding:
- Does flipping a bit to 0 in both `a` and `b` count as two flips? → Yes.

> 💡 **Key insight.** OR works on each bit position separately, so the minimum flips is the sum of the minimum flips per position.

**Brute force →** Try flip sets → exponential. Per bit → O(32).

**Steps**
1. Look at the lowest bits `x`, `y`, `z` of `a`, `b`, `c`.
2. `z == 0`: both must be 0 → add `x + y`.
3. `z == 1`: need at least one 1 → add 1 if both are 0.
4. Shift all three right and repeat.

**Complexity**

- **Time:** O(32).
- **Space:** O(1).

In [53]:
def min_flips(a: int, b: int, c: int) -> int:
    """Bit flips making a | b == c."""
    flips = 0
    while a or b or c:
        x, y, z = a & 1, b & 1, c & 1
        # c bit 0: clear every set bit; c bit 1: need at least one set bit
        flips += (x + y) if z == 0 else int(x == y == 0)
        a, b, c = a >> 1, b >> 1, c >> 1
    return flips


assert min_flips(2, 6, 5) == 3
assert min_flips(4, 2, 7) == 1
assert min_flips(1, 2, 3) == 0
print(min_flips(2, 6, 5))

3


---
# 5) Design

**In plain English.** Design questions ask for a class whose operations each hit a target
cost. List the operations and their required costs first, then pick one structure per
need and make them point at each other.

> **Analogy.** A library: a catalogue (hash map) to find a book instantly, shelves
> (an array) to grab a random one, and a returns cart (a queue) for order of arrival.
> Each structure does one job well.

| Need | Structure |
|---|---|
| O(1) lookup by key | hash map |
| O(1) random access / uniform random pick | array (+ map from value to index) |
| O(1) at both ends, sliding windows | deque / ring buffer |
| O(1) reorder / remove given a node | doubly linked list |
| O(log n) min / max | heap |
| order queries ("largest ≤ t") | sorted list + bisect |
| hierarchy / prefixes | trie |

**Recognition signals**

- "design / implement a class", "all operations in O(1)"
- "in the last N seconds", "moving average", "circular queue"
- "iterator", "peek", "flatten"
- "file system", "paths"

**Common bugs**

- Removing from the middle of an array in O(n) — swap with the last element first.
- Forgetting to update the *moved* element's index in the map.
- Full vs empty ambiguity in a ring buffer (track `size` explicitly).
- Iterators that consume an element inside `has_next` without buffering it.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| LRU / LFU caches | map + doubly linked list (covered with linked lists) |
| Heap-based designs | median finder, Twitter feed — O(log n) operations |
| Algorithm questions | a design question is graded per *operation*, not on one input |

**Learning path.** Map + array with swap-delete (RandomizedSet) → time windows with a queue
(Hit Counter, Moving Average) → ring buffer → hashing from scratch → sparse
representation → iterators with lookahead and lazy flattening → aggregation by key
(Underground) → a trie-shaped file system.

## 5.1 Data-structure design

Before coding, write `operation → required cost → structure`. Most bugs come from an
operation that silently costs O(n) (removing from a list, scanning a window).

### 5.1.1 [380. Insert Delete GetRandom O(1)](https://leetcode.com/problems/insert-delete-getrandom-o1/) — ★★ · Medium

**Problem.** `insert`, `remove` and `get_random` (uniform), all average O(1).  
**Signal.** Array for random access + hash map value → index; delete by swapping with the last element.  
*New here: Swap-with-last deletion to keep an array compact.*

**Fine print.** Ask before coding:
- Can a value be inserted twice? → No, the second insert returns `False`.
- Will `get_random` only be called when the set is non-empty? → Yes.

> 💡 **Key insight.** Removing from the end of an array is O(1), so move the last element into the removed slot and fix its index in the map.

**Brute force →** A set alone can't pick uniformly at random in O(1); a list alone removes in O(n). Map + array → O(1) for all three.

**Steps**
1. `values` array and `index` map value → position.
2. `insert`: append and record the index.
3. `remove`: copy the last value into the removed slot, update its index, pop the end, delete the key.
4. `get_random`: `random.choice(values)`.

**Complexity**

- **Time:** O(1) average per operation.
- **Space:** O(n).

In [54]:
class RandomizedSet:
    """Set with O(1) insert, remove and uniform sampling."""

    def __init__(self) -> None:
        # compact array for sampling; value -> position for lookups
        self.values: list[int] = []
        self.index: dict[int, int] = {}

    def insert(self, val: int) -> bool:
        if val in self.index:
            return False
        self.index[val] = len(self.values)
        self.values.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.index:
            return False
        # move the last value into the hole and fix its index
        i, last = self.index[val], self.values[-1]
        self.values[i], self.index[last] = last, i
        # removing from the end is O(1)
        self.values.pop()
        del self.index[val]
        return True

    def get_random(self) -> int:
        return random.choice(self.values)


rs = RandomizedSet()
assert rs.insert(1) and not rs.remove(2) and rs.insert(2)
assert rs.get_random() in (1, 2)
assert rs.remove(1) and not rs.insert(2) and rs.get_random() == 2
print(rs.values)

[2]


### 5.1.2 [362. Design Hit Counter](https://leetcode.com/problems/design-hit-counter/) — ★★ · Medium · Premium

**Problem.** `hit(timestamp)` and `get_hits(timestamp)` = hits in the past 300 seconds; timestamps are non-decreasing.  
**Signal.** Queue of timestamps; evict entries older than the window.  
*New here: A time window maintained by a queue.*

**Fine print.** Ask before coding:
- Are timestamps non-decreasing? → Yes.
- Can several hits share a timestamp? → Yes.
- Follow-up: very many hits per second? → Use 300 buckets.

> 💡 **Key insight.** Timestamps arrive in order, so the oldest hits are always at the front of a queue and can be evicted there.

**Brute force →** Store all hits and count the recent ones each time → O(n) per query. Queue eviction → amortised O(1).

**Steps**
1. `hit` appends the timestamp.
2. `get_hits` pops from the front while the oldest is ≤ `timestamp − 300`.
3. Return the queue length.

**Complexity**

- **Time:** Amortised O(1) — each hit is appended and popped once.
- **Space:** O(hits in the window).

In [55]:
class HitCounter:
    """Hits within the last 300 seconds."""

    def __init__(self) -> None:
        # timestamps in arrival (non-decreasing) order
        self.hits: deque[int] = deque()

    def hit(self, timestamp: int) -> None:
        self.hits.append(timestamp)

    def get_hits(self, timestamp: int) -> int:
        # evict hits that fell out of the 300-second window
        while self.hits and self.hits[0] <= timestamp - 300:
            self.hits.popleft()
        return len(self.hits)


hc = HitCounter()
for t in (1, 2, 3):
    hc.hit(t)
assert hc.get_hits(4) == 3
hc.hit(300)
assert hc.get_hits(300) == 4 and hc.get_hits(301) == 3
print(hc.get_hits(301))

3


### 5.1.3 [346. Moving Average from Data Stream](https://leetcode.com/problems/moving-average-from-data-stream/) — ★ · Easy · Premium

**Problem.** Average of the last `size` values of a stream.  
**Signal.** Fixed window → deque with a running sum.  
*New here: A fixed window with a running sum, as a class.*

**Fine print.** Ask before coding:
- Fewer than `size` values so far? → Average what you have.

> 💡 **Key insight.** Adding one value and removing the oldest updates the sum in O(1); the deque remembers which value is oldest.

**Brute force →** Sum the last `size` values each call → O(size). Running sum → O(1).

**Steps**
1. Append the value and add it to the total.
2. If the window is too big, pop the oldest and subtract it.
3. Return `total / len(window)`.

**Complexity**

- **Time:** O(1) per value.
- **Space:** O(size).

In [56]:
class MovingAverage:
    """Average over a sliding window."""

    def __init__(self, size: int) -> None:
        self.size = size
        self.window: deque[int] = deque()
        self.total = 0

    def next(self, val: int) -> float:
        self.window.append(val)
        self.total += val
        # keep only the last `size` values
        if len(self.window) > self.size:
            self.total -= self.window.popleft()
        return self.total / len(self.window)


ma = MovingAverage(3)
out = [ma.next(v) for v in [1, 10, 3, 5]]
assert out == [1.0, 5.5, 14 / 3, 6.0]
print(out)

[1.0, 5.5, 4.666666666666667, 6.0]


### 5.1.4 [622. Design Circular Queue](https://leetcode.com/problems/design-circular-queue/) — ★ · Medium

**Problem.** Fixed-capacity FIFO queue on an array with `en_queue`, `de_queue`, `front`, `rear`.  
**Signal.** Ring buffer: head index + size, positions taken modulo capacity.  
*New here: The ring buffer.*

**Fine print.** Ask before coding:
- Operations on an empty or full queue? → Return `False` or −1 as specified.
- Can I use a built-in queue? → No.

> 💡 **Key insight.** With a head index and a size, the tail position is `(head + size) % capacity`, so the array wraps around and nothing ever shifts.

**Brute force →** A Python list with `pop(0)` → O(n) dequeue. Ring buffer → O(1).

**Steps**
1. Store `data`, `head` and `size`.
2. `en_queue`: if not full, write at `(head + size) % k` and grow `size`.
3. `de_queue`: if not empty, advance `head` modulo k and shrink `size`.
4. `front` = `data[head]`, `rear` = `data[(head + size − 1) % k]`.

**Complexity**

- **Time:** O(1) per operation.
- **Space:** O(k).

⚠️ **Trap.** Without `size`, `head == tail` is ambiguous (full or empty).

In [57]:
class MyCircularQueue:
    """Ring-buffer queue."""

    def __init__(self, k: int) -> None:
        self.data = [0] * k
        # size separates "full" from "empty"
        self.head = self.size = 0

    def en_queue(self, value: int) -> bool:
        if self.size == len(self.data):
            return False
        # the slot just after the last element, wrapping around
        self.data[(self.head + self.size) % len(self.data)] = value
        self.size += 1
        return True

    def de_queue(self) -> bool:
        if not self.size:
            return False
        self.head = (self.head + 1) % len(self.data)
        self.size -= 1
        return True

    def front(self) -> int:
        return self.data[self.head] if self.size else -1

    def rear(self) -> int:
        return (
            self.data[(self.head + self.size - 1) % len(self.data)] if self.size else -1
        )


q = MyCircularQueue(3)
assert [q.en_queue(v) for v in (1, 2, 3, 4)] == [True, True, True, False]
assert q.rear() == 3 and q.de_queue() and q.en_queue(4) and q.rear() == 4
assert q.front() == 2
print(q.data)

[4, 2, 3]


### 5.1.5 [706. Design HashMap](https://leetcode.com/problems/design-hashmap/) — ★ · Easy

**Problem.** Hash map with `put`, `get`, `remove` without built-in hash tables.  
**Signal.** Array of buckets (separate chaining).  
*New here: How a hash map works inside (buckets + chaining).*

**Fine print.** Ask before coding:
- Keys and values in `0..10⁶`? → Yes.
- Does `put` on an existing key update it? → Yes.
- Can I use a built-in dict? → No.

> 💡 **Key insight.** A hash function sends each key to one bucket; with enough buckets each holds only a few pairs, so scanning a bucket is O(1) on average.

**Brute force →** One list of pairs → O(n) per op. A direct array of size 10⁶ → O(1) but wasteful. Buckets → O(1) average with modest memory.

**Steps**
1. Bucket = `key % size` (prime size spreads keys).
2. `put`: update the pair if the key exists, else append.
3. `get`: scan the bucket.
4. `remove`: filter the key out of its bucket.

**Complexity**

- **Time:** O(1) average.
- **Space:** O(n + size).

In [58]:
class MyHashMap:
    """Separate-chaining hash map."""

    def __init__(self, size: int = 1009) -> None:
        # a prime number of buckets spreads keys evenly
        self.buckets: list[list[list[int]]] = [[] for _ in range(size)]

    def _bucket(self, key: int) -> list[list[int]]:
        return self.buckets[key % len(self.buckets)]

    def put(self, key: int, value: int) -> None:
        bucket = self._bucket(key)
        # existing key: overwrite in place
        for pair in bucket:
            if pair[0] == key:
                pair[1] = value
                return
        bucket.append([key, value])

    def get(self, key: int) -> int:
        for k, v in self._bucket(key):
            if k == key:
                return v
        return -1

    def remove(self, key: int) -> None:
        bucket = self._bucket(key)
        bucket[:] = [pair for pair in bucket if pair[0] != key]


hm = MyHashMap()
hm.put(1, 1)
hm.put(2, 2)
assert hm.get(1) == 1 and hm.get(3) == -1
hm.put(2, 1)
assert hm.get(2) == 1
hm.remove(2)
assert hm.get(2) == -1
print("ok")

ok


### 5.1.6 [1570. Dot Product of Two Sparse Vectors](https://leetcode.com/problems/dot-product-of-two-sparse-vectors/) — ★ · Medium · Premium

**Problem.** Compute dot products of sparse vectors efficiently.  
**Signal.** Store only non-zero entries as `index → value`.  
*New here: Sparse representation.*

**Fine print.** Ask before coding:
- Is only one vector sparse? → Then iterate over the sparse one; ask about it as the follow-up.
- Same length? → Yes.

> 💡 **Key insight.** Zero entries contribute nothing, so store only non-zeros and iterate over the smaller set of them.

**Brute force →** Multiply all n positions → O(n). Sparse dicts → O(min(nnz₁, nnz₂)).

**Steps**
1. Store `{index: value}` for non-zero entries.
2. Iterate over the smaller dict.
3. Multiply by the matching entry in the other (0 if absent) and sum.

**Complexity**

- **Time:** O(min(nnz₁, nnz₂)).
- **Space:** O(nnz).

In [59]:
class SparseVector:
    """Sparse vector keyed by non-zero index."""

    def __init__(self, nums: list[int]) -> None:
        # zeros are never stored
        self.entries = {i: x for i, x in enumerate(nums) if x}

    def dot_product(self, other: "SparseVector") -> int:
        # walk the vector with fewer non-zeros
        small, big = sorted((self.entries, other.entries), key=len)
        return sum(x * big.get(i, 0) for i, x in small.items())


v1, v2 = SparseVector([1, 0, 0, 2, 3]), SparseVector([0, 3, 0, 4, 0])
assert v1.dot_product(v2) == 8
assert (
    SparseVector([0, 1, 0, 0, 2, 0, 0]).dot_product(SparseVector([1, 0, 0, 0, 3, 0, 4]))
    == 6
)
print(v1.dot_product(v2))

8


### 5.1.7 [284. Peeking Iterator](https://leetcode.com/problems/peeking-iterator/) — ★ · Medium

**Problem.** Wrap an iterator to support `peek()` in addition to `next()` and `has_next()`.  
**Signal.** Cache one element ahead.  
*New here: One-element lookahead with a sentinel for "exhausted".*

**Fine print.** Ask before coding:
- Can `peek` be called many times in a row? → Yes, without advancing.
- Generic types? → Ask; the follow-up asks for it.

> 💡 **Key insight.** If you always hold the next element already fetched, `peek` just returns it and `next` returns it and fetches another.

**Brute force →** Convert the whole iterator to a list → O(n) memory and not lazy. A one-element buffer → O(1).

**Steps**
1. Pre-fetch the first element (or a sentinel at the end).
2. `peek` returns the buffer.
3. `next` returns the buffer and pre-fetches again.
4. `has_next` checks for the sentinel.

**Complexity**

- **Time:** O(1) per operation.
- **Space:** O(1).

In [60]:
class PeekingIterator:
    """Iterator with one-element lookahead."""

    # unique marker for "no more elements" (None could be a real value)
    _END = object()

    def __init__(self, iterator) -> None:
        self.it = iterator
        self._next = next(self.it, self._END)

    def peek(self):
        return self._next

    def next(self):
        value = self._next
        # fetch the following element ahead of time
        self._next = next(self.it, self._END)
        return value

    def has_next(self) -> bool:
        return self._next is not self._END


pi = PeekingIterator(iter([1, 2, 3]))
assert pi.next() == 1 and pi.peek() == 2 and pi.next() == 2 and pi.next() == 3
assert not pi.has_next()
print("ok")

ok


### 5.1.8 [341. Flatten Nested List Iterator](https://leetcode.com/problems/flatten-nested-list-iterator/) — ★ · Medium

**Problem.** Iterate over the integers of an arbitrarily nested list.  
**Signal.** Stack of iterators (lazy) — or flatten eagerly with recursion.  
*New here: A stack of iterators for lazy traversal of nested data.*

**Fine print.** Ask before coding:
- Can lists be empty or nested inside empty lists? → Yes, so `has_next` must skip them.
- Lazy or eager? → Lazy is the better answer.

> 💡 **Key insight.** The stack of iterators mirrors the recursion stack, so the next integer can be found on demand without flattening everything first.

**Brute force →** Flatten the whole structure up front → O(total) memory and time before the first `next`. Lazy stack → amortised O(1).

**Steps**
1. Stack starts with an iterator over the top list.
2. `has_next`: pull from the top iterator; exhausted → pop; a list → push its iterator; an integer → buffer it.
3. `next`: ensure the buffer is filled, then return it.

**Complexity**

- **Time:** Amortised O(1) per element.
- **Space:** O(depth).

In [61]:
class NestedIterator:
    """Lazy flattening iterator."""

    def __init__(self, nested: list) -> None:
        # one iterator per open list level
        self.stack = [iter(nested)]
        self.buffer: list[int] = []

    def has_next(self) -> bool:
        # advance until an integer is found or nothing is left
        while not self.buffer and self.stack:
            item = next(self.stack[-1], None)
            # this level is exhausted
            if item is None:
                self.stack.pop()
            # descend into a nested list
            elif isinstance(item, list):
                self.stack.append(iter(item))
            else:
                self.buffer.append(item)
        return bool(self.buffer)

    def next(self) -> int:
        self.has_next()
        return self.buffer.pop()


it = NestedIterator([[1, 1], 2, [1, [4, [6]]], []])
out = []
while it.has_next():
    out.append(it.next())
assert out == [1, 1, 2, 1, 4, 6]
print(out)

[1, 1, 2, 1, 4, 6]


### 5.1.9 [1396. Design Underground System](https://leetcode.com/problems/design-underground-system/) — ★ · Medium

**Problem.** Check-in/check-out events; `get_average_time(start, end)` over completed trips.  
**Signal.** Two hash maps: open trips by customer, and totals by `(start, end)`.  
*New here: Aggregating on the fly instead of storing raw events.*

**Fine print.** Ask before coding:
- Is every trip's check-out after its check-in, by the same card? → Yes.
- Is A→B different from B→A? → Yes.
- Does a query always have at least one trip? → Yes.

> 💡 **Key insight.** An average only needs a running sum and a count per route, so update them at check-out and never store individual trips.

**Brute force →** Store all trips and average on query → O(trips) per query. Running totals → O(1).

**Steps**
1. Check-in: `open[id] = (station, time)`.
2. Check-out: pop it, add the duration to `totals[(start, end)]` and increment the count.
3. Average = sum / count.

**Complexity**

- **Time:** O(1) per operation.
- **Space:** O(customers + routes).

In [62]:
class UndergroundSystem:
    """Average trip times between stations."""

    def __init__(self) -> None:
        # customer -> (station, time) of the trip in progress
        self.open: dict[int, tuple[str, int]] = {}
        # (start, end) -> [total time, trip count]
        self.totals: dict[tuple[str, str], list[int]] = defaultdict(lambda: [0, 0])

    def check_in(self, cid: int, station: str, t: int) -> None:
        self.open[cid] = (station, t)

    def check_out(self, cid: int, station: str, t: int) -> None:
        start, t0 = self.open.pop(cid)
        # fold the finished trip into its route's totals
        total = self.totals[(start, station)]
        total[0] += t - t0
        total[1] += 1

    def get_average_time(self, start: str, end: str) -> float:
        total, count = self.totals[(start, end)]
        return total / count


u = UndergroundSystem()
u.check_in(45, "Leyton", 3)
u.check_in(32, "Paradise", 8)
u.check_out(45, "Waterloo", 15)
u.check_out(32, "Cambridge", 22)
assert u.get_average_time("Paradise", "Cambridge") == 14.0
u.check_in(10, "Leyton", 24)
u.check_out(10, "Waterloo", 38)
assert u.get_average_time("Leyton", "Waterloo") == 13.0
print(u.get_average_time("Leyton", "Waterloo"))

13.0


### 5.1.10 [588. Design In-Memory File System](https://leetcode.com/problems/design-in-memory-file-system/) — ★ · Hard · Premium

**Problem.** `ls`, `mkdir`, `add_content_to_file` and `read_content_from_file` on absolute paths.  
**Signal.** A trie of path components; each node is a directory or a file.  
*New here: A trie whose keys are path components.*

**Fine print.** Ask before coding:
- Is `ls` on a file just the file name? → Yes.
- Output sorted lexicographically? → Yes.
- Does `add_content_to_file` create the file if it does not exist? → Yes, and appends otherwise.

> 💡 **Key insight.** A path is a sequence of names, so directories form a trie; a file is a node that has content.

**Brute force →** A flat dict from full path to content → `ls` must scan every path, O(total). The trie → O(path length).

**Steps**
1. Each node: `children` dict and `content` (`None` for directories).
2. `_walk(path)`: follow (creating as needed) one node per component.
3. `ls`: a file returns its own name; a directory returns its sorted children.
4. `add_content` appends to the node's content; `read` returns it.

**Complexity**

- **Time:** O(path length) per operation (+ O(k log k) to sort k children in `ls`).
- **Space:** O(total path components + content).

In [63]:
class FileSystem:
    """In-memory hierarchical file system."""

    def __init__(self) -> None:
        # content None = directory, a string = file
        self.root: dict = {"children": {}, "content": None}

    def _walk(self, path: str) -> dict:
        node = self.root
        # one trie level per path component, created on demand
        for part in filter(None, path.split("/")):
            node = node["children"].setdefault(part, {"children": {}, "content": None})
        return node

    def ls(self, path: str) -> list[str]:
        node = self._walk(path)
        # a file lists only itself
        if node["content"] is not None:
            return [path.split("/")[-1]]
        return sorted(node["children"])

    def mkdir(self, path: str) -> None:
        self._walk(path)

    def add_content_to_file(self, path: str, content: str) -> None:
        node = self._walk(path)
        node["content"] = (node["content"] or "") + content

    def read_content_from_file(self, path: str) -> str:
        return self._walk(path)["content"]


fs = FileSystem()
assert fs.ls("/") == []
fs.mkdir("/a/b/c")
fs.add_content_to_file("/a/b/c/d", "hello")
assert fs.ls("/") == ["a"] and fs.ls("/a/b/c") == ["d"] and fs.ls("/a/b/c/d") == ["d"]
assert fs.read_content_from_file("/a/b/c/d") == "hello"
print("ok")

ok


---
# Summary

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Farthest reach | jump games | track max reachable index / BFS window | O(n) |
| Restart on failure | gas station, Kadane-like scans | a failed prefix rules out every start inside it | O(n) |
| Last occurrence | partition labels | stretch the window to the last index of every letter | O(n) |
| Range of states | wildcard parentheses | track `[lo, hi]` of possible counts | O(n) |
| Two-pass constraints | candy | fix left neighbours, then right | O(n) |
| Sort by key + greedy | two cities, queue by height, cookies | exchange argument picks the order | O(n log n) |
| Coverage doubling | patching array | add the smallest missing value | O(m + log n) |
| Merge intervals | merge / insert / free time | sort by start, extend the last | O(n log n) |
| Activity selection | removals, arrows | sort by end, keep earliest finishers | O(n log n) |
| Active count | meeting rooms, query intervals | heap of ends | O(n log n) |
| Digit math | reverse, palindrome number, multiply strings | `divmod(x, 10)`, carry arrays | O(digits) |
| Fast power / sieve / gcd | pow, primes, string gcd | squaring, mark multiples from p², `s+t == t+s` | O(log n) / O(n log log n) |
| XOR tricks | single / missing numbers | pairs cancel, split by a differing bit | O(n) |
| Bit DP | counting bits | `bits[i >> 1] + (i & 1)` | O(n) |
| Design | O(1) operations | combine map + array / deque / ring buffer | O(1) |